# Tester performance de nos méthodes 

In [25]:
import pandas as pd
import os, glob
import numpy as np
from collections import defaultdict
import json
import cv2
# import glob, os
import matplotlib.pyplot as plt
from skimage import io
# from scipy.ndimage import median_filter
from scipy.ndimage import convolve1d

### Les fonctions de nos méthodes 

#### Pour fusionner les segments trouvés 

In [125]:
from collections import defaultdict
def fusionner_segments(segments_etendus):
    """
    Prend une liste de segments sous la forme [x, y_start, y_end],
    les regroupe par colonne (x), puis fusionne les segments qui se chevauchent ou se touchent.
    """
    if not segments_etendus:
        return {}

    # 1. Regrouper les segments par colonne (x)
    dict_intermediaire = defaultdict(list)
    for segment in segments_etendus:
        # Ici on lit simplement les 3 valeurs de ta liste [x, y_start, y_end]
        x = segment[0]
        y_start = segment[1]
        y_end = segment[2]
        dict_intermediaire[x].append((y_start, y_end))

    dict_final = {}

    # 2. Traiter chaque colonne indépendamment
    for x, liste_segments in dict_intermediaire.items():
        
        if len(liste_segments) <= 1:
            dict_final[x] = liste_segments
            continue
            
        segments_tries = sorted(liste_segments, key=lambda coord: coord[0])
        segments_fusionnes = [segments_tries[0]]
        
        for segment_actuel in segments_tries[1:]:
            dernier_segment_valide = segments_fusionnes[-1]
            
            debut_actuel, fin_actuelle = segment_actuel
            debut_dernier, fin_derniere = dernier_segment_valide
            
            if debut_actuel <= fin_derniere:
                nouvelle_fin = max(fin_derniere, fin_actuelle)
                segments_fusionnes[-1] = (debut_dernier, nouvelle_fin)
            else:
                segments_fusionnes.append(segment_actuel)
                
        dict_final[x] = segments_fusionnes

    return dict_final

In [124]:
# plus actuelle version
def fusionner_segments_(segments):
    """
    Fonction utilitaire : si deux petits défauts s'étendent et finissent 
    par se rentrer dedans, on les fusionne en un seul grand défaut.
    """
    if not segments: return []
    
    # Grouper par colonne (x)
    dict_cols = {}
    for seg in segments:
        x, y_s, y_e = seg[0][0], seg[0][1], seg[1][1]
        if x not in dict_cols: dict_cols[x] = []
        dict_cols[x].append([y_s, y_e])
        
    segments_fusion = []
    for x, intervalles in dict_cols.items():
        intervalles.sort(key=lambda v: v[0]) # Trier de haut en bas
        fusion = [intervalles[0]]
        
        for courant in intervalles[1:]:
            dernier = fusion[-1]
            # Si les segments se chevauchent ou se touchent
            if courant[0] <= dernier[1] + 1:
                dernier[1] = max(dernier[1], courant[1])
            else:
                fusion.append(courant)
                
        for y_s, y_e in fusion:
            segments_fusion.append([(x, y_s), (x, y_e)])
            
    return segments_fusion

#### CFAR_band + Region Growing

In [28]:
def detecter_et_mesurer_defauts_complet(image, hauteur_bande=50, train_cells=4, guard_cells=2, multiplicateur_rupture=3.5):
    """
    Pipeline complet de détection et mesure des colonnes défectueuses (entières et fragmentées).
    
    1. Découpe l'image en bandes et applique un filtre CA-CFAR pour trouver des "graines" de défauts.
    2. Prolonge ces graines vers le haut et le bas (Region Growing) jusqu'à une rupture d'intensité.
    3. Fusionne les segments qui se chevauchent et formate le résultat final.
    
    Args:
        image (numpy.ndarray): L'image 16-bits en entrée.
        hauteur_bande (int): Taille de la bande horizontale pour le CFAR (défaut: 50).
        train_cells (int): Nombre de cellules d'entraînement de chaque côté (défaut: 4).
        guard_cells (int): Nombre de cellules de garde de chaque côté (défaut: 2).
        multiplicateur_rupture (float): Tolérance pour le seuil de rupture lors de la croissance (défaut: 3.5).
        
    Returns:
        dict: Dictionnaire formaté {x: [(y1_start, y1_end), (y2_start, y2_end), ...]} 

    """
    height, width = image.shape

    # =========================================================
    # ETAPE 1 : DETECTION DES GRAINES (CFAR par bandes)
    # =========================================================
    segments_initiaux = []
    
    # --- Paramètres du CFAR ---
    num_train_side = train_cells 
    num_guard_side = guard_cells  
    taille_fenetre = (num_train_side * 2) + (num_guard_side * 2) + 1 
    noyau = np.zeros(taille_fenetre)
    noyau[:num_train_side] = 1.0  
    noyau[-num_train_side:] = 1.0 
    noyau = noyau / (num_train_side * 2)

    # On parcourt l'image de haut en bas, en sautant de 'hauteur_bande' en 'hauteur_bande'
    for y_start in range(0, height, hauteur_bande):
        y_end = min(y_start + hauteur_bande, height) # min() pour ne pas déborder à la fin
        
        # On extrait la sous-image (la bande horizontale)
        bande = image[y_start:y_end, :]
        
        # On calcule la projection médiane UNIQUEMENT sur cette bande
        projection_bande = np.median(bande, axis=0)
        
        # On applique le CFAR
        bruit_de_fond_local = convolve1d(projection_bande, noyau, mode='nearest')
        
        # Tolérance locale pour cette bande spécifique
        ecart_type_bande = np.std(projection_bande)
        tolerance = 3 * ecart_type_bande  
        
        seuil_haut = bruit_de_fond_local + tolerance
        seuil_bas = bruit_de_fond_local - tolerance
        
        # Détection pour cette bande
        colonnes_detectees = np.where(
            (projection_bande > seuil_haut) | 
            (projection_bande < seuil_bas)
        )[0]
        
        # On enregistre les résultats sous forme de segments initiaux (graines)
        for x in colonnes_detectees:
            segments_initiaux.append([(x, y_start), (x, y_end)])


    # =========================================================
    # ETAPE 2 : CROISSANCE DE REGION (Region Growing)
    # =========================================================
    segments_etendus = []

    for segment in segments_initiaux:
        # On force la conversion en entier natif python pour la suite
        x = int(segment[0][0])
        y_start = int(segment[0][1])
        y_end = int(segment[1][1])
        
        colonne = image[:, x].astype(np.float32) # pour chaque x, on prend tous les px de la col
        
        # 1. Calculer ce qu'est un "saut normal" / et seuil sur cette colonne
        sauts_verticaux = np.abs(np.diff(colonne))  # On regarde la dérivée absolue (la différence entre chaque pixel et le suivant (de la col))

        bruit_normal = np.median(sauts_verticaux) # on calc les sauts normaux (sur le défaut, ou sur le vrai paysage de l'img) en prennant médiane de tous les sauts 
        ecart_sauts = np.std(sauts_verticaux)
        
        seuil_rupture = bruit_normal + (multiplicateur_rupture * ecart_sauts) # calc marche qu'on considère trop grande (fin du défaut)

        # 2. Prolonger vers le HAUT (on remonte la colonne)
        while y_start > 0: # boucle tant qu'on a pas atteint le haut de l'image
            # Quelle est la taille de la marche pour monter sur le pixel du dessus ?
            saut_haut = np.abs(colonne[y_start] - colonne[y_start - 1]) # diff entre px et celui d'avnat 
            
            if saut_haut < seuil_rupture:
                # Intensité similaire : on est toujours dans le défaut
                y_start -= 1 # on continue de rémonter 
            else:
                # BOUM ! Rupture forte d'intensité, on a trouvé le bord supérieur.
                break

        # 3. Prolonger vers le BAS (on descend la colonne)
        while y_end < (height - 1): # boucle tant qu'on a pas atteint le bas de l'image
            # Quelle est la taille de la marche pour descendre sur le pixel du dessous ?
            saut_bas = np.abs(colonne[y_end] - colonne[y_end + 1]) # diff entre px et celui d'après 
            
            if saut_bas < seuil_rupture:
                y_end += 1 # on continue de descendre 
            else:
                break # sinon saut trop gros, on a trouvé bord inf
                
        # On stocke les coordonnées étendues sous forme de liste simple pour la fusion
        segments_etendus.append([x, y_start, y_end])

          
    return fusionner_segments(segments_etendus) # ne pas oublier de fusionner au cas où y'a des chevauchements


#### Local Thresholding with Variance, Median or Mean 

In [29]:
def local_threshold(image, taille_fenetre=50, metrique ='Moyenne', facteur_std=1.0,visual_graph = False):

    height = image.shape[0]

    if metrique == 'Moyenne':
        metrique_ = np.mean(image, axis=0)
        label = 'Moyenne'
    elif metrique == 'Mediane':
        metrique_ = np.median(image, axis=0)
        label = 'Mediane'
    elif metrique == 'Variance':
        metrique_ = np.std(image, axis=0)
        label = 'Variance'
    else : 
        print('Error de metrique')

    filtre = np.ones(taille_fenetre) / taille_fenetre
    tendance_locale = convolve1d(metrique_, filtre, mode='reflect')
    
    #Marge de tolérance
    marge_tolerance = facteur_std * np.std(metrique_)
    
    # Le seuil n'est plus un simple nombre, c'est un tableau de la même taille que l'image !
    threshold_haut = tendance_locale + marge_tolerance
    threshold_bas = tendance_locale - marge_tolerance
    
    # Détection bilatérale avec l'écart absolu (np.abs)
    defect_columns = np.where(np.abs(metrique_ - tendance_locale) > marge_tolerance)[0]

    predit_dict = {}
    for col in defect_columns:
        # On convertit 'col' en int natif Python pour éviter les soucis avec JSON/Dictionnaires
        # Et on indique que le défaut s'étend de la ligne 0 jusqu'en bas (height)
        predit_dict[int(col)] = [(0, height)]

    if visual_graph:
        # Printing results    
        print(f"Analyse basée sur : {label}")
        print(f"Marge de tolérance (+ 2*std) : {marge_tolerance:.2f}")
        print(f"Nombre de colonnes dépassant le seuil local : {len(defect_columns)}")
        print(f"Index de ces colonnes : {defect_columns}")

        # Affichage du graphe
        plt.figure(figsize=(12, 6))
        
        # Tracer la courbe de toutes les variances/moyennes
        plt.plot(metrique_, label=label, color='#1f77b4', linewidth=1.5, alpha=0.8)
        
        # Tracer la courbe du threshold local
        plt.plot(threshold_haut, color='orange', linestyle='--', linewidth=2, label='Seuil Haut (+2 std)')
        plt.plot(threshold_bas, color='green', linestyle='--', linewidth=2, label='Seuil Bas (-2 std)')
        
        # Mettre un point rouge sur le graphe pour chaque "pire" colonne
        plt.scatter(defect_columns, metrique_[defect_columns], color='red', zorder=5, label='Colonnes critiques')

        # Personnalisation du graphe
        plt.title(f'{label} des intensités par colonne (avec Seuil Local)', fontsize=14)
        plt.xlabel('Index de la colonne (Position X dans l\'image)', fontsize=12)
        plt.ylabel(f'{label}', fontsize=12)
        plt.legend()
        plt.grid(True, linestyle=':', alpha=0.7)
        
        plt.tight_layout()
        plt.show()
    return predit_dict

#### Nouvelle version simplifiée juste pour la moyenne 

In [122]:
def local_threshold(image, taille_fenetre=50, facteur_std=1.0):

    # height = image.shape[0]  plus besoin ici !
    
    metrique_ = np.mean(image, axis=0)
    label = 'Moyenne'
    
    filtre = np.ones(taille_fenetre) / taille_fenetre
    tendance_locale = convolve1d(metrique_, filtre, mode='reflect')
    
    # Marge de tolérance
    marge_tolerance = facteur_std * np.std(metrique_)
    
    # Le seuil n'est plus un simple nombre, c'est un tableau de la même taille que l'image !
    threshold_haut = tendance_locale + marge_tolerance
    threshold_bas = tendance_locale - marge_tolerance
    
    # Détection bilatérale avec l'écart absolu (np.abs)
    defect_columns = np.where(np.abs(metrique_ - tendance_locale) > marge_tolerance)[0]

    # On renvoie juste une liste de 'int' pour chaque cols 
    return [int(col) for col in defect_columns]

In [97]:
img = io.imread ('train/VGA/sequence_1/low dyn with columns 1/frame_0000.png')
print(local_threshold(image=img, metrique='Moyenne'))

{395: [(0, 512)], 450: [(0, 512)]}


### Local thresholding par bandes 

In [30]:
from collections import defaultdict

def local_threshold_bandes(image, num_bandes=4, taille_fenetre=50, metrique='Moyenne', facteur_std=1.0, visual_graph=False):
    height, width = image.shape
    
    # On calcule la hauteur d'une bande selon le nb que y'en a 
    hauteur_bande = height // num_bandes
    
    # defaultdict permet d'ajouter facilement des tuples à une liste existante
    predit_dict = defaultdict(list)
    
    # Stockage pour le graphe si demandé
    debug_data = []

    for b in range(num_bandes):
        # 1. Définir les limites y de la bande
        y_start = b * hauteur_bande
        # Si c'est la dernière bande, on va jusqu'au bout de l'image (pour ne pas rater de pixels à cause de l'arrondi)
        y_end = height if b == num_bandes - 1 else (b + 1) * hauteur_bande
        
        # 2. Extraire la bande
        bande = image[y_start:y_end, :]

        # 3. Calcul de la métrique sur CETTE bande
        if metrique == 'Moyenne':
            metrique_ = np.mean(bande, axis=0)
            label = 'Moyenne'
        elif metrique == 'Mediane':
            metrique_ = np.median(bande, axis=0)
            label = 'Mediane'
        elif metrique == 'Variance':
            metrique_ = np.std(bande, axis=0)
            label = 'Variance'
        else: 
            print('Erreur de métrique')
            return {}

        # 4. Lissage (Tendance locale)
        filtre = np.ones(taille_fenetre) / taille_fenetre
        tendance_locale = convolve1d(metrique_, filtre, mode='reflect')
        
        # 5. Marge de tolérance (ajustée avec facteur_std !)
        marge_tolerance = np.std(metrique_) * facteur_std
        
        # 6. Détection bilatérale
        defect_columns = np.where(np.abs(metrique_ - tendance_locale) > marge_tolerance)[0]

        # 7. Enregistrement dans le dictionnaire
        for col in defect_columns:
            # On stocke les coordonnées (y_start, y_end) spécifiques à cette bande
            predit_dict[int(col)].append((y_start, y_end))
            
        if visual_graph:
            debug_data.append((b, metrique_, tendance_locale, marge_tolerance, defect_columns))

    if visual_graph:
        # On crée des sous-graphes pour chaque bande
        fig, axes = plt.subplots(num_bandes, 1, figsize=(12, 3 * num_bandes), sharex=True)
        if num_bandes == 1:
            axes = [axes] # Sécurité si on ne demande qu'une seule bande
            
        fig.suptitle(f'Analyse par bandes basée sur : {label} (Fenêtre={taille_fenetre}, Std={facteur_std})', fontsize=16)

        for ax, data in zip(axes, debug_data):
            b, met_, tend_, marge_, def_cols = data
            
            ax.plot(met_, label=f'{label} Bande {b+1}', color='#1f77b4', linewidth=1.5, alpha=0.8)
            ax.plot(tend_ + marge_, color='orange', linestyle='--', linewidth=1.5, label=f'+{facteur_std} std')
            ax.plot(tend_ - marge_, color='green', linestyle='--', linewidth=1.5, label=f'-{facteur_std} std')
            ax.scatter(def_cols, met_[def_cols], color='red', zorder=5)
            
            ax.set_ylabel(f'Bande {b+1}')
            ax.grid(True, linestyle=':', alpha=0.7)
            ax.legend(loc='upper right')
            
        axes[-1].set_xlabel('Index de la colonne (Position X dans l\'image)')
        plt.tight_layout()
        plt.show()

    # On convertit le defaultdict en dict classique avant de le renvoyer
    return dict(predit_dict)

### Notre fonction d'évéluation de perf

#### Pour load un dossier, le JSON, et recup les défauts gt pour une certaine image

In [31]:
def load_images(folder='train',type='VGA',sequence='sequence_1', dyn='low dyn with columns 1', force_gray=False):
    """
    function that load a folder of images

    args : 
    folder : folder type
    type : type of the frame (HD, VGA, SXGA)
    sequence : sequence_1, sequence_2, sequence_3
    dyn : low dyn with columns 1, low dyn with columns 2, low dyn with columns 3
    force_gray : bool to load in gray (one array)

    return :
    list of the images of the folder
    """
    images = []

    chemin_recherche = os.path.join(folder, type, sequence, dyn, '*.png')

    fichiers_trouves = sorted(glob.glob(chemin_recherche))

    if force_gray == True:
        for image_path in fichiers_trouves:
            img = io.imread(image_path, as_gray=True)
            images.append(img)
    else:
        for image_path in fichiers_trouves:
            img = io.imread(image_path)
            images.append(img)
            
    return images


def load_json(file_path):
    """
    Loads a JSON file and returns its content.

    Args:
        file_path (str): The path to the JSON file.

    Returns:
        list/dict: The parsed JSON data.
    """
    with open(file_path, "r", encoding="utf-8") as file:
        data = json.load(file)
    
    return data


def get_defect_coordinates(json_data, image_number):
    """
    Parses JSON data to extract defect coordinates for a specific image number.

    Args:
        json_data (list): The list of dictionaries loaded from the JSON file.
        image_number (str or int): The specific image number to look for in the JSON.

    Returns:
        dict: A dictionary where keys are x-coordinates (columns) and values 
              are dictionaries containing lists of 'start' and 'stop' y-coordinates.
    """
    defect_dict = {}
    
    for item in json_data:
        value = item.get('signal', {}).get(str(image_number))
        
        if value is not None:
            for x_coord in item.get('x_coord', []):
                
                defect_dict[x_coord] = {
                    'ycords': (list(zip(item.get('y_start', []),item.get('y_stop', [])))),
                    'type': item.get('name', [])
                }

    return defect_dict

#### L'évaluation : comparaison des dico gt/res image par image 

In [32]:
def evaluate_detection(vrai_dict, predit_dict, printing = False):
    """
    Évalue la détection avec correspondance exacte (sans tolérance).
    Force la conversion des clés en entiers (int) pour éviter les erreurs de type string/int.
    Calcule et retourne la Précision, le Rappel et le F1-Score.
    """
    vrais_positifs = 0
    faux_negatifs = 0
    faux_positifs = 0
    
    # On convertit toutes les colonnes prédites en int pour être sûr du format
    colonnes_predites_int = {int(x) for x in predit_dict.keys()}
    colonnes_predites_utilisees = set()

    # =========================================================
    # 1. Vérifier ce qui est bien détecté et ce qui est oublié
    # =========================================================
    for x_vrai_raw, infos_vrai in vrai_dict.items():
        x_vrai = int(x_vrai_raw) # On force la vérité terrain en int
        
        # On vérifie la correspondance exacte
        if x_vrai in colonnes_predites_int:
            vrais_positifs += 1
            colonnes_predites_utilisees.add(x_vrai)
        else:
            faux_negatifs += 1

    # =========================================================
    # 2. Vérifier les fausses alarmes (Faux Positifs)
    # =========================================================
    for x_pred in colonnes_predites_int:
        # Si la colonne prédite n'a pas matché avec une vraie colonne
        if x_pred not in colonnes_predites_utilisees:
            faux_positifs += 1

    # =========================================================
    # 3. Calcul des statistiques finales
    # =========================================================
    precision = vrais_positifs / (vrais_positifs + faux_positifs) if (vrais_positifs + faux_positifs) > 0 else 0
    rappel = vrais_positifs / (vrais_positifs + faux_negatifs) if (vrais_positifs + faux_negatifs) > 0 else 0
    
    # Calcul du F1-Score
    if (precision + rappel) > 0:
        f1_score = 2 * (precision * rappel) / (precision + rappel)
    else:
        f1_score = 0
    
    if printing:
        print("\n--- RÉSULTATS DE LA DÉTECTION (Correspondance exacte) ---")
        print(f"Vrais Positifs (Bien trouvés)  : {vrais_positifs}")
        print(f"Faux Négatifs (Oubliés)        : {faux_negatifs}")
        print(f"Faux Positifs (Fausses alarmes): {faux_positifs}")
        print(f"Précision (Fiabilité)          : {precision*100:.1f}%")
        print(f"Rappel (Taux de découverte)    : {rappel*100:.1f}%")
        print(f"F1-Score (Score global)        : {f1_score*100:.1f}%")

    return vrais_positifs, faux_positifs, faux_negatifs, f1_score

#### L'évaluation de la methode sur toute une séquence 

In [ ]:
def evaluate_sequence_detection(methode_detection=local_threshold, 
                                folder='train', img_type='VGA', sequence='sequence_1', dyn='low dyn with columns 1', update=False):
    """
    Évalue la détection sur toute une séquence d'images.
    dataset_name : Le nom du dataset (qui servira à trouver les images et le JSON)
    methode_detection : La fonction à utiliser pour prédire les défauts
    """
    # 1. Charger les images (assure-toi que load_images gère bien le format attendu)

    chiffre = int(dyn.split()[-1])

    data = load_images(folder=folder, type=img_type, sequence=sequence, dyn=dyn, force_gray=False)
    json_name = f'{img_type}_{sequence}_config_{chiffre}'
    
    # recup les JSON 
    chemin_json = f'results/{json_name}.json'
    json_data = load_json(chemin_json)
    
    # Initialisation des compteurs GLOBAUX
    total_vp = 0 # true positive
    total_fp = 0 # false positive
    total_fn = 0 # false negative

    # 3. Boucle sur chaque image
    for i in range(len(data)):
        vrai_dict = get_defect_coordinates(json_data, i) # on recup dico des vrais defauts gt
        predit_dict = methode_detection(data[i]) # on recup dico des défauts qu'on trouve avec notre methode
        vp, fp, fn, f1_image = evaluate_detection(vrai_dict, predit_dict) # on compare les 2 dicos 
                                                                        # possible d'afficher les res de la detection à chaque image (ATTENTION peut polluer le terminal si trop d'images)
                                                                        # en mettant printing = True 
        # On ajoute aux compteurs globaux
        total_vp += vp 
        total_fp += fp
        total_fn += fn

    # =========================================================
    # 4. Calcul des scores globaux de la séquence
    # =========================================================
    precision_seq = total_vp / (total_vp + total_fp) if (total_vp + total_fp) > 0 else 0
    rappel_seq = total_vp / (total_vp + total_fn) if (total_vp + total_fn) > 0 else 0
    
    if (precision_seq + rappel_seq) > 0:
        f1_score_seq = 2 * (precision_seq * rappel_seq) / (precision_seq + rappel_seq)
    else:
        f1_score_seq = 0

    # 5. Affichage des résultats
    if update is True : 
        print("\n" + "="*50)
        print(f"RÉSULTATS GLOBAUX - SÉQUENCE : type : {img_type} sequence : {sequence} dyn : {dyn}")
        print("="*50)
        print(f"Total Vrais Positifs (VP) : {total_vp}")
        print(f"Total Faux Négatifs (FN)  : {total_fn} (Oublis)")
        print(f"Total Faux Positifs (FP)  : {total_fp} (Fausses alarmes)")
        print("-" * 50)
        print(f"Précision de la séquence  = {precision_seq*100:.2f}%")
        print(f"Recall de la séquence     = {rappel_seq*100:.2f}%")
        print(f"F1_score de la séquence   = {f1_score_seq*100:.2f}%")
        print("="*50 + "\n")
    
    return precision_seq, rappel_seq, f1_score_seq

### Fonctio pour évaluer sur les différentes seq d'un type 

In [79]:
def evaluate_type_detection(methode_detection, 
                          folder='train', 
                          img_type='VGA', 
                          sequences=['sequence_1', 'sequence_2', 'sequence_3'], 
                          dyns=['low dyn with columns 1', 'low dyn with columns 2', 'low dyn with columns 3'], 
                          update=False):
    """
    Évalue la détection sur plusieurs séquences ET plusieurs types de défauts (dyns).
    Cumule les VP, FP, et FN sur l'ensemble avant de calculer le score final.
    """
    # Compteurs GLOBAUX
    total_vp = 0
    total_fp = 0
    total_fn = 0

    # 1. Boucle sur les séquences
    for seq in sequences:
        
        # 2. Boucle sur les dynamiques (les différents types de colonnes défectueuses)
        for dyn in dyns:
            
            # Récupérer le numéro du défaut (ex: "3" pour "columns 3")
            chiffre = int(dyn.split()[-1])
            
            # Charger les images pour cette combinaison (séquence + dyn)
            data = load_images(folder=folder, type=img_type, sequence=seq, dyn=dyn, force_gray=False)
            
            # Charger le JSON correspondant
            json_name = f'{img_type}_{seq}_config_{chiffre}'
            chemin_json = f'results/{json_name}.json'
            
            try:
                json_data = load_json(chemin_json)
            except FileNotFoundError:
                print(f"Attention : Fichier {chemin_json} introuvable. Combinaison {seq} / {dyn} ignorée.")
                continue
                
            # 3. Boucle sur chaque image de ce dossier spécifique
            for i in range(len(data)):
                vrai_dict = get_defect_coordinates(json_data, i)
                predit_dict = methode_detection(data[i])
                vp, fp, fn, _ = evaluate_detection(vrai_dict, predit_dict)
                
                total_vp += vp
                total_fp += fp
                total_fn += fn

    # =========================================================
    # 4. Calcul des scores GLOBAUX finaux (sur toutes les images de tous les dossiers)
    # =========================================================
    precision_globale = total_vp / (total_vp + total_fp) if (total_vp + total_fp) > 0 else 0
    rappel_global = total_vp / (total_vp + total_fn) if (total_vp + total_fn) > 0 else 0
    
    if (precision_globale + rappel_global) > 0:
        f1_score_global = 2 * (precision_globale * rappel_global) / (precision_globale + rappel_global)
    else:
        f1_score_global = 0

    # 5. Affichage des résultats globaux
    if update is True: 
        print("\n" + "="*60)
        print(f"RÉSULTATS GLOBAUX - TYPE : {img_type}")
        print(f"Séquences : {sequences}")
        print(f"Dynamiques : {dyns}")
        print("="*60)
        print(f"Total Vrais Positifs (VP) : {total_vp}")
        print(f"Total Faux Négatifs (FN)  : {total_fn} (Oublis)")
        print(f"Total Faux Positifs (FP)  : {total_fp} (Fausses alarmes)")
        print("-" * 60)
        print(f"Précision globale         = {precision_globale*100:.2f}%")
        print(f"Recall global             = {rappel_global*100:.2f}%")
        print(f"F1_score global           = {f1_score_global*100:.2f}%")
        print("="*60 + "\n")
    
    return precision_globale, rappel_global, f1_score_global

### Comment 
ATTENTION prend bcp de temps de recharger img à chaque fois 

## Test des différentes méthode avec notre fctn d'évaluation de perf

In [34]:
from functools import partial

In [37]:
print("Test de Local Thresholding avec la Métrique de l'Ecart-Type")
evaluate_sequence_detection(
    methode_detection= partial(local_threshold, metrique='Variance'),
    folder="train",
    img_type="VGA",
    sequence="sequence_1",
    dyn="low dyn with columns 3"
)
print(50*"==")
print("Test de Local Thresholding avec la Métrique de la Médiane")
evaluate_sequence_detection(
    methode_detection=partial(local_threshold, metrique='Mediane'),
    folder="train",
    img_type="VGA",
    sequence="sequence_1",
    dyn="low dyn with columns 3"
)
print(50*"==")
print("Test de Local Thresholding avec la Métrique de la Moyenne")
evaluate_sequence_detection(
    methode_detection= partial (local_threshold, metrique='Moyenne'),
    folder="train",
    img_type="VGA",
    sequence="sequence_1",
    dyn="low dyn with columns 3"
)
print(50*"==")
print("Test de CFAR_band et Region Growing")
evaluate_sequence_detection(
    methode_detection= detecter_et_mesurer_defauts_complet,
    folder="train",
    img_type="VGA",
    sequence="sequence_1",
    dyn="low dyn with columns 3"
)

Test de Local Thresholding avec la Métrique de l'Ecart-Type

RÉSULTATS GLOBAUX - SÉQUENCE : type : VGA sequence : sequence_1 dyn : low dyn with columns 3
Total Vrais Positifs (VP) : 1513
Total Faux Négatifs (FN)  : 1454 (Oublis)
Total Faux Positifs (FP)  : 53292 (Fausses alarmes)
--------------------------------------------------
Précision de la séquence  = 2.76%
Recall de la séquence     = 50.99%
F1_score de la séquence   = 5.24%

Test de Local Thresholding avec la Métrique de la Médiane

RÉSULTATS GLOBAUX - SÉQUENCE : type : VGA sequence : sequence_1 dyn : low dyn with columns 3
Total Vrais Positifs (VP) : 2582
Total Faux Négatifs (FN)  : 385 (Oublis)
Total Faux Positifs (FP)  : 0 (Fausses alarmes)
--------------------------------------------------
Précision de la séquence  = 100.00%
Recall de la séquence     = 87.02%
F1_score de la séquence   = 93.06%

Test de Local Thresholding avec la Métrique de la Moyenne

RÉSULTATS GLOBAUX - SÉQUENCE : type : VGA sequence : sequence_1 dyn : low

(0.9247538677918424, 0.886417256488035, 0.905179831354328)

### TEST pour HD

In [88]:
print("Test de LT_moy pour seq 1 col 3 HD")
evaluate_sequence_detection(
    methode_detection= partial (local_threshold, metrique='Moyenne', taille_fenetre=24, facteur_std=1.25),
    folder="train",
    img_type="HD",
    sequence="sequence_1",
    dyn="low dyn with columns 3", 
    update=True
)

Test de Local Thresholding avec la Métrique de la Moyenne

RÉSULTATS GLOBAUX - SÉQUENCE : type : HD sequence : sequence_1 dyn : low dyn with columns 3
Total Vrais Positifs (VP) : 2755
Total Faux Négatifs (FN)  : 5337 (Oublis)
Total Faux Positifs (FP)  : 4806 (Fausses alarmes)
--------------------------------------------------
Précision de la séquence  = 36.44%
Recall de la séquence     = 34.05%
F1_score de la séquence   = 35.20%



(0.36436979235550854, 0.3404597132970835, 0.3520091995144701)

In [93]:
print("Test de LT_moy pour seq 1 col 1 HD")
evaluate_sequence_detection(
    methode_detection= partial (local_threshold, metrique='Moyenne', taille_fenetre=24, facteur_std=1.25),
    folder="train",
    img_type="HD",
    sequence="sequence_1",
    dyn="low dyn with columns 3", 
    update=True
)

Test de LT_moy pour seq 1 col 1 HD

RÉSULTATS GLOBAUX - SÉQUENCE : type : HD sequence : sequence_1 dyn : low dyn with columns 3
Total Vrais Positifs (VP) : 2755
Total Faux Négatifs (FN)  : 5337 (Oublis)
Total Faux Positifs (FP)  : 4806 (Fausses alarmes)
--------------------------------------------------
Précision de la séquence  = 36.44%
Recall de la séquence     = 34.05%
F1_score de la séquence   = 35.20%



(0.36436979235550854, 0.3404597132970835, 0.3520091995144701)

### Comment 
C'est dramatiquement nul purée

### TEST avec CFAR pour voir si c'est mieux 

In [ ]:
print(50*"==")
print("Test de CFAR_band_RG sur seq1 col 3")
evaluate_sequence_detection(
    methode_detection= detecter_et_mesurer_defauts_complet,
    folder="train",
    img_type="HD",
    sequence="sequence_1",
    dyn="low dyn with columns 3"
)

#### Test du Local Thresholding par bandes pour Var, Mean, Median

In [40]:
print(50*"==")
print("Test de Local Thresholding par bandes avec la Métrique Variance")
evaluate_sequence_detection(
    methode_detection= partial(local_threshold_bandes, metrique='Variance'),
    folder="train",
    img_type="VGA",
    sequence="sequence_1",
    dyn="low dyn with columns 3"
)
print(50*"==")
print("Test de Local Thresholding par bandes avec la Métrique Mediane")
evaluate_sequence_detection(
    methode_detection= partial(local_threshold_bandes, metrique='Mediane'),
    folder="train",
    img_type="VGA",
    sequence="sequence_1",
    dyn="low dyn with columns 3"
)
print(50*"==")
print("Test de Local Thresholding par bandes avec la Métrique Moyenne")
evaluate_sequence_detection(
    methode_detection= partial(local_threshold_bandes, metrique='Moyenne'),
    folder="train",
    img_type="VGA",
    sequence="sequence_1",
    dyn="low dyn with columns 3"
)

Test de Local Thresholding par bandes avec la Métrique Variance

RÉSULTATS GLOBAUX - SÉQUENCE : type : VGA sequence : sequence_1 dyn : low dyn with columns 3
Total Vrais Positifs (VP) : 2395
Total Faux Négatifs (FN)  : 572 (Oublis)
Total Faux Positifs (FP)  : 232215 (Fausses alarmes)
--------------------------------------------------
Précision de la séquence  = 1.02%
Recall de la séquence     = 80.72%
F1_score de la séquence   = 2.02%

Test de Local Thresholding par bandes avec la Métrique Mediane

RÉSULTATS GLOBAUX - SÉQUENCE : type : VGA sequence : sequence_1 dyn : low dyn with columns 3
Total Vrais Positifs (VP) : 2853
Total Faux Négatifs (FN)  : 114 (Oublis)
Total Faux Positifs (FP)  : 12517 (Fausses alarmes)
--------------------------------------------------
Précision de la séquence  = 18.56%
Recall de la séquence     = 96.16%
F1_score de la séquence   = 31.12%

Test de Local Thresholding par bandes avec la Métrique Moyenne

RÉSULTATS GLOBAUX - SÉQUENCE : type : VGA sequence : seq

(0.3626303739506487, 0.9609032692955848, 0.5265490811709299)

# col 3 de seq 1

In [43]:
import optuna

### Optimisation des params pour local thresholding with band for metrique MEDIAN and in col 3 of seq 1

In [44]:
# ==========================================
# 1. LA FONCTION OBJECTIF POUR OPTUNA
# ==========================================
def objective(trial):
    """ 
    cette fonction suggère des val de param à tester -> donne le f1_score avec ce param 
    """
    # 1. Optuna suggère les paramètres à tester pour cet essai
    taille_fenetre_test = trial.suggest_int("taille_fenetre", 10, 150)
    facteur_std_test = trial.suggest_float("facteur_std", 1.0, 4.0)
    num_bandes_test = trial.suggest_int("num_bandes", 1, 10)
    
    # 2. On prépare notre fonction de détection avec ces nouveaux paramètres
    # On utilise partial pour "geler" les paramètres sans exécuter la fonction
    methode_test = partial(
        local_threshold_bandes, 
        taille_fenetre=taille_fenetre_test,
        facteur_std=facteur_std_test,
        num_bandes=num_bandes_test,
        metrique='Mediane',
        visual_graph=False # Surtout pas de graphes pendant l'optimisation !
    )
    
    # 3. On appelle TA fonction d'évaluation (en mode silencieux)
    precision, recall, f1_score = evaluate_sequence_detection(
        methode_detection=methode_test,
        folder='train',
        img_type='VGA',
        sequence='sequence_1',
        dyn='low dyn with columns 3',
        update=False # <-- On cache les prints pour ne pas polluer le terminal
    )
    
    # 4. On renvoie uniquement le F1-score à Optuna car c'est ce qu'il doit maximiser
    return f1_score


# ==========================================
# 2. LANCEMENT DE L'OPTIMISATION
# ==========================================
if __name__ == "__main__":
    print("🚀 Lancement de l'optimisation Bayésienne avec Optuna...")
    
    # On crée l'étude en demandant de maximiser la valeur renvoyée (le f1_score)
    study = optuna.create_study(direction="maximize")
    
    # On lance 50 essais (tu peux monter à 100 ou 200 si ça va vite)
    study.optimize(objective, n_trials=50)

    # ==========================================
    # 3. AFFICHAGE DU RÉSULTAT FINAL
    # ==========================================
    print("\n" + 50*"🌟")
    print("OPTIMISATION TERMINÉE")
    print(50*"🌟")
    print(f"Meilleur F1-Score atteint : {study.best_value * 100:.2f}%")
    print("Paramètres parfaits pour ce score :")
    for cle, valeur in study.best_params.items():
        print(f"  -> {cle} : {valeur}")

[I 2026-05-29 15:41:58,718] A new study created in memory with name: no-name-7158ca5d-75a9-45c2-8220-00b0fec7f2c4


🚀 Lancement de l'optimisation Bayésienne avec Optuna...


[I 2026-05-29 15:42:04,147] Trial 0 finished with value: 0.6881588624662908 and parameters: {'taille_fenetre': 75, 'facteur_std': 1.3347533859318066, 'num_bandes': 4}. Best is trial 0 with value: 0.6881588624662908.
[I 2026-05-29 15:42:09,245] Trial 1 finished with value: 0.6417944609750514 and parameters: {'taille_fenetre': 18, 'facteur_std': 3.50451516556664, 'num_bandes': 2}. Best is trial 0 with value: 0.6881588624662908.
[I 2026-05-29 15:42:14,683] Trial 2 finished with value: 0.6809500489715965 and parameters: {'taille_fenetre': 80, 'facteur_std': 1.8493224536783721, 'num_bandes': 8}. Best is trial 0 with value: 0.6881588624662908.
[I 2026-05-29 15:42:19,565] Trial 3 finished with value: 0.9163845633039946 and parameters: {'taille_fenetre': 37, 'facteur_std': 2.367835591965206, 'num_bandes': 10}. Best is trial 3 with value: 0.9163845633039946.
[I 2026-05-29 15:42:24,823] Trial 4 finished with value: 0.9483153995413653 and parameters: {'taille_fenetre': 124, 'facteur_std': 2.43964


🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟
OPTIMISATION TERMINÉE
🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟
Meilleur F1-Score atteint : 95.26%
Paramètres parfaits pour ce score :
  -> taille_fenetre : 25
  -> facteur_std : 2.122570050763343
  -> num_bandes : 4


### Optimisation LT_moy_opti 

In [63]:
# ==========================================
# 1. LA FONCTION OBJECTIF POUR OPTUNA
# ==========================================
def objective(trial):
    """ 
    cette fonction suggère des val de param à tester -> donne le f1_score avec ce param 
    """
    # 1. Optuna suggère les paramètres à tester pour cet essai
    taille_fenetre_test = trial.suggest_int("taille_fenetre", 10, 150)
    facteur_std_test = trial.suggest_float("facteur_std", 1.0, 4.0)
    # num_bandes_test = trial.suggest_int("num_bandes", 1, 10)
    
    # 2. On prépare notre fonction de détection avec ces nouveaux paramètres
    # On utilise partial pour "geler" les paramètres sans exécuter la fonction
    methode_test = partial(
        local_threshold, 
        taille_fenetre=taille_fenetre_test,
        facteur_std=facteur_std_test,
        # num_bandes=num_bandes_test,
        metrique='Moyenne',
        visual_graph=False # Surtout pas de graphes pendant l'optimisation !
    )
    
    # 3. On appelle TA fonction d'évaluation (en mode silencieux)
    precision, recall, f1_score = evaluate_sequence_detection(
        methode_detection=methode_test,
        folder='train',
        img_type='VGA',
        sequence='sequence_1',
        dyn='low dyn with columns 3',
        update=False # <-- On cache les prints pour ne pas polluer le terminal
    )
    
    # 4. On renvoie uniquement le F1-score à Optuna car c'est ce qu'il doit maximiser
    return f1_score


# ==========================================
# 2. LANCEMENT DE L'OPTIMISATION
# ==========================================
if __name__ == "__main__":
    print("🚀 Lancement de l'optimisation Bayésienne avec Optuna...")
    
    # On crée l'étude en demandant de maximiser la valeur renvoyée (le f1_score)
    study = optuna.create_study(direction="maximize")
    
    # On lance 50 essais (tu peux monter à 100 ou 200 si ça va vite)
    study.optimize(objective, n_trials=50)

    # ==========================================
    # 3. AFFICHAGE DU RÉSULTAT FINAL
    # ==========================================
    print("\n" + 50*"🌟")
    print("OPTIMISATION TERMINÉE")
    print(50*"🌟")
    print(f"Meilleur F1-Score atteint : {study.best_value * 100:.2f}%")
    print("Paramètres parfaits pour ce score :")
    for cle, valeur in study.best_params.items():
        print(f"  -> {cle} : {valeur}")

[I 2026-06-01 13:18:35,537] A new study created in memory with name: no-name-183c62b6-8afc-4d99-ac9d-0e5b0512088e


🚀 Lancement de l'optimisation Bayésienne avec Optuna...


[I 2026-06-01 13:18:39,535] Trial 0 finished with value: 0.9275257545635279 and parameters: {'taille_fenetre': 88, 'facteur_std': 1.2003909804761892}. Best is trial 0 with value: 0.9275257545635279.
[I 2026-06-01 13:18:43,136] Trial 1 finished with value: 0.7892266884309324 and parameters: {'taille_fenetre': 62, 'facteur_std': 2.5524921862402294}. Best is trial 0 with value: 0.9275257545635279.
[I 2026-06-01 13:18:46,409] Trial 2 finished with value: 0.9452541770351939 and parameters: {'taille_fenetre': 142, 'facteur_std': 1.1093133636660188}. Best is trial 2 with value: 0.9452541770351939.
[I 2026-06-01 13:18:50,618] Trial 3 finished with value: 0.7911998370340192 and parameters: {'taille_fenetre': 34, 'facteur_std': 2.4924428214501377}. Best is trial 2 with value: 0.9452541770351939.
[I 2026-06-01 13:18:53,826] Trial 4 finished with value: 0.6894876325088338 and parameters: {'taille_fenetre': 43, 'facteur_std': 3.1047104246875294}. Best is trial 2 with value: 0.9452541770351939.
[I 2


🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟
OPTIMISATION TERMINÉE
🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟
Meilleur F1-Score atteint : 95.84%
Paramètres parfaits pour ce score :
  -> taille_fenetre : 150
  -> facteur_std : 1.0016419046068692


#### Comapraiosn avec LT_moy 

In [65]:
print("Test de Local Thresholding avec la Métrique de la Moyenne")
evaluate_sequence_detection(
    methode_detection= partial (local_threshold, metrique='Moyenne'),
    folder="train",
    img_type="VGA",
    sequence="sequence_1",
    dyn="low dyn with columns 3", 
    update=True
)

Test de Local Thresholding avec la Métrique de la Moyenne

RÉSULTATS GLOBAUX - SÉQUENCE : type : VGA sequence : sequence_1 dyn : low dyn with columns 3
Total Vrais Positifs (VP) : 2696
Total Faux Négatifs (FN)  : 271 (Oublis)
Total Faux Positifs (FP)  : 0 (Fausses alarmes)
--------------------------------------------------
Précision de la séquence  = 100.00%
Recall de la séquence     = 90.87%
F1_score de la séquence   = 95.21%



(1.0, 0.9086619480957195, 0.9521455059155923)

### Optimisation des params pour local thresholding with band for metrique MOYENNE (LT_moy_band_opti) and in col 3 of seq 1

In [45]:
# ==========================================
# 1. LA FONCTION OBJECTIF POUR OPTUNA
# ==========================================
def objective(trial):
    """ 
    cette fonction suggère des val de param à tester -> donne le f1_score avec ce param 
    """
    # 1. Optuna suggère les paramètres à tester pour cet essai
    taille_fenetre_test = trial.suggest_int("taille_fenetre", 10, 150)
    facteur_std_test = trial.suggest_float("facteur_std", 1.0, 4.0)
    num_bandes_test = trial.suggest_int("num_bandes", 1, 10)
    
    # 2. On prépare notre fonction de détection avec ces nouveaux paramètres
    # On utilise partial pour "geler" les paramètres sans exécuter la fonction
    methode_test = partial(
        local_threshold_bandes, 
        taille_fenetre=taille_fenetre_test,
        facteur_std=facteur_std_test,
        num_bandes=num_bandes_test,
        metrique='Moyenne',
        visual_graph=False # Surtout pas de graphes pendant l'optimisation !
    )
    
    # 3. On appelle TA fonction d'évaluation (en mode silencieux)
    precision, recall, f1_score = evaluate_sequence_detection(
        methode_detection=methode_test,
        folder='train',
        img_type='VGA',
        sequence='sequence_1',
        dyn='low dyn with columns 3',
        update=False # <-- On cache les prints pour ne pas polluer le terminal
    )
    
    # 4. On renvoie uniquement le F1-score à Optuna car c'est ce qu'il doit maximiser
    return f1_score


# ==========================================
# 2. LANCEMENT DE L'OPTIMISATION
# ==========================================
if __name__ == "__main__":
    print("🚀 Lancement de l'optimisation Bayésienne avec Optuna...")
    
    # On crée l'étude en demandant de maximiser la valeur renvoyée (le f1_score)
    study = optuna.create_study(direction="maximize")
    
    # On lance 50 essais (tu peux monter à 100 ou 200 si ça va vite)
    study.optimize(objective, n_trials=50)

    # ==========================================
    # 3. AFFICHAGE DU RÉSULTAT FINAL
    # ==========================================
    print("\n" + 50*"🌟")
    print("OPTIMISATION TERMINÉE")
    print(50*"🌟")
    print(f"Meilleur F1-Score atteint : {study.best_value * 100:.2f}%")
    print("Paramètres parfaits pour ce score :")
    for cle, valeur in study.best_params.items():
        print(f"  -> {cle} : {valeur}")

[I 2026-05-29 15:54:05,451] A new study created in memory with name: no-name-4c5f5c5f-2e5d-4afc-be70-74769205a71f


🚀 Lancement de l'optimisation Bayésienne avec Optuna...


[I 2026-05-29 15:54:09,733] Trial 0 finished with value: 0.3534261977637579 and parameters: {'taille_fenetre': 131, 'facteur_std': 1.3856330036305051, 'num_bandes': 7}. Best is trial 0 with value: 0.3534261977637579.
[I 2026-05-29 15:54:13,594] Trial 1 finished with value: 0.8108216432865731 and parameters: {'taille_fenetre': 150, 'facteur_std': 2.400884328141485, 'num_bandes': 1}. Best is trial 1 with value: 0.8108216432865731.
[I 2026-05-29 15:54:17,117] Trial 2 finished with value: 0.6632124352331606 and parameters: {'taille_fenetre': 112, 'facteur_std': 3.215308114830308, 'num_bandes': 1}. Best is trial 1 with value: 0.8108216432865731.
[I 2026-05-29 15:54:20,498] Trial 3 finished with value: 0.9457254345512593 and parameters: {'taille_fenetre': 16, 'facteur_std': 2.6707931633148547, 'num_bandes': 10}. Best is trial 3 with value: 0.9457254345512593.
[I 2026-05-29 15:54:23,807] Trial 4 finished with value: 0.9560516219044297 and parameters: {'taille_fenetre': 108, 'facteur_std': 1.9


🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟
OPTIMISATION TERMINÉE
🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟
Meilleur F1-Score atteint : 96.57%
Paramètres parfaits pour ce score :
  -> taille_fenetre : 36
  -> facteur_std : 1.4439935316800787
  -> num_bandes : 3


#### Comparaison avec LT_band pas optimisée 

In [58]:
print("Test de Local Thresholding par bandes avec la Métrique Moyenne")
evaluate_sequence_detection(
    methode_detection= partial(local_threshold_bandes, metrique='Moyenne'),
    folder="train",
    img_type="VGA",
    sequence="sequence_1",
    dyn="low dyn with columns 3", 
    update=True
)

Test de Local Thresholding par bandes avec la Métrique Moyenne

RÉSULTATS GLOBAUX - SÉQUENCE : type : VGA sequence : sequence_1 dyn : low dyn with columns 3
Total Vrais Positifs (VP) : 2851
Total Faux Négatifs (FN)  : 116 (Oublis)
Total Faux Positifs (FP)  : 5011 (Fausses alarmes)
--------------------------------------------------
Précision de la séquence  = 36.26%
Recall de la séquence     = 96.09%
F1_score de la séquence   = 52.65%



(0.3626303739506487, 0.9609032692955848, 0.5265490811709299)

#### CFAR_band_opti

In [66]:
# ==========================================
# 1. LA FONCTION OBJECTIF POUR OPTUNA
# ==========================================
def objective(trial):
    """ 
    cette fonction suggère des val de param à tester -> donne le f1_score avec ce param 
    """
    # 1. Optuna suggère les paramètres à tester pour cet essai
    hauteur_bande = trial.suggest_int("hauteur_bande", 10, 150)
    train_cells = trial.suggest_int("train_cells", 4, 10)
    guard_cells = trial.suggest_int("guard_cells", 2, 5)
    multiplicateur_rupture = trial.suggest_float("multiplicateur_rupture", 2.0, 5.0)    

    # 2. On prépare notre fonction de détection avec ces nouveaux paramètres
    # On utilise partial pour "geler" les paramètres sans exécuter la fonction
    methode_test = partial(
        detecter_et_mesurer_defauts_complet, 
        hauteur_bande=hauteur_bande,
        train_cells=train_cells,
        guard_cells=guard_cells,
        multiplicateur_rupture=multiplicateur_rupture,
    )
    
    # 3. On appelle TA fonction d'évaluation (en mode silencieux)
    precision, recall, f1_score = evaluate_sequence_detection(
        methode_detection=methode_test,
        folder='train',
        img_type='VGA',
        sequence='sequence_1',
        dyn='low dyn with columns 3',
        update=False # <-- On cache les prints pour ne pas polluer le terminal
    )
    
    # 4. On renvoie uniquement le F1-score à Optuna car c'est ce qu'il doit maximiser
    return f1_score


# ==========================================
# 2. LANCEMENT DE L'OPTIMISATION
# ==========================================
if __name__ == "__main__":
    print("🚀 Lancement de l'optimisation Bayésienne avec Optuna...")
    
    # On crée l'étude en demandant de maximiser la valeur renvoyée (le f1_score)
    study = optuna.create_study(direction="maximize")
    
    # On lance 50 essais (tu peux monter à 100 ou 200 si ça va vite)
    study.optimize(objective, n_trials=50)

    # ==========================================
    # 3. AFFICHAGE DU RÉSULTAT FINAL
    # ==========================================
    print("\n" + 50*"🌟")
    print("OPTIMISATION TERMINÉE")
    print(50*"🌟")
    print(f"Meilleur F1-Score atteint : {study.best_value * 100:.2f}%")
    print("Paramètres parfaits pour ce score :")
    for cle, valeur in study.best_params.items():
        print(f"  -> {cle} : {valeur}")

[I 2026-06-01 13:25:11,232] A new study created in memory with name: no-name-a4a71304-79d9-4af6-9d5e-b61a595b4ad8


🚀 Lancement de l'optimisation Bayésienne avec Optuna...


[I 2026-06-01 13:25:18,997] Trial 0 finished with value: 0.9071379071379072 and parameters: {'hauteur_bande': 84, 'train_cells': 6, 'guard_cells': 2, 'multiplicateur_rupture': 3.0231037194379327}. Best is trial 0 with value: 0.9071379071379072.
[I 2026-06-01 13:25:28,079] Trial 1 finished with value: 0.9074844074844074 and parameters: {'hauteur_bande': 84, 'train_cells': 10, 'guard_cells': 4, 'multiplicateur_rupture': 4.934623468241921}. Best is trial 1 with value: 0.9074844074844074.
[I 2026-06-01 13:25:36,478] Trial 2 finished with value: 0.939978563772776 and parameters: {'hauteur_bande': 110, 'train_cells': 8, 'guard_cells': 5, 'multiplicateur_rupture': 3.948571211490638}. Best is trial 2 with value: 0.939978563772776.
[I 2026-06-01 13:25:42,500] Trial 3 finished with value: 0.9365698086463501 and parameters: {'hauteur_bande': 119, 'train_cells': 8, 'guard_cells': 2, 'multiplicateur_rupture': 2.1598742929294623}. Best is trial 2 with value: 0.939978563772776.
[I 2026-06-01 13:25:48


🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟
OPTIMISATION TERMINÉE
🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟
Meilleur F1-Score atteint : 94.26%
Paramètres parfaits pour ce score :
  -> hauteur_bande : 116
  -> train_cells : 8
  -> guard_cells : 3
  -> multiplicateur_rupture : 3.2979306504745933


#### Compa avec CFAR_band

In [67]:
print("Test de CFAR_band et Region Growing")
evaluate_sequence_detection(
    methode_detection= detecter_et_mesurer_defauts_complet,
    folder="train",
    img_type="VGA",
    sequence="sequence_1",
    dyn="low dyn with columns 3", 
    update = True
)

Test de CFAR_band et Region Growing

RÉSULTATS GLOBAUX - SÉQUENCE : type : VGA sequence : sequence_1 dyn : low dyn with columns 3
Total Vrais Positifs (VP) : 2630
Total Faux Négatifs (FN)  : 337 (Oublis)
Total Faux Positifs (FP)  : 214 (Fausses alarmes)
--------------------------------------------------
Précision de la séquence  = 92.48%
Recall de la séquence     = 88.64%
F1_score de la séquence   = 90.52%



(0.9247538677918424, 0.886417256488035, 0.905179831354328)

# col 3 de seq 2

### Optimisation LT_moy_opti 

In [55]:
# ==========================================
# 1. LA FONCTION OBJECTIF POUR OPTUNA
# ==========================================
def objective(trial):
    """ 
    cette fonction suggère des val de param à tester -> donne le f1_score avec ce param 
    """
    # 1. Optuna suggère les paramètres à tester pour cet essai
    taille_fenetre_test = trial.suggest_int("taille_fenetre", 10, 150)
    facteur_std_test = trial.suggest_float("facteur_std", 1.0, 4.0)
    # num_bandes_test = trial.suggest_int("num_bandes", 1, 10) pas besoin ici puisque c'est la version pas avec bandes
    
    # 2. On prépare notre fonction de détection avec ces nouveaux paramètres
    # On utilise partial pour "geler" les paramètres sans exécuter la fonction
    methode_test = partial(
        local_threshold, 
        taille_fenetre=taille_fenetre_test,
        facteur_std=facteur_std_test,
        # num_bandes=num_bandes_test,
        metrique='Moyenne',
        visual_graph=False # Surtout pas de graphes pendant l'optimisation !
    )
    
    # 3. On appelle TA fonction d'évaluation (en mode silencieux)
    precision, recall, f1_score = evaluate_sequence_detection(
        methode_detection=methode_test,
        folder='train',
        img_type='VGA',
        sequence='sequence_2',
        dyn='low dyn with columns 3',
        update=False # <-- On cache les prints pour ne pas polluer le terminal
    )
    
    # 4. On renvoie uniquement le F1-score à Optuna car c'est ce qu'il doit maximiser
    return f1_score


# ==========================================
# 2. LANCEMENT DE L'OPTIMISATION
# ==========================================
if __name__ == "__main__":
    print("🚀 Lancement de l'optimisation Bayésienne avec Optuna...")
    
    # On crée l'étude en demandant de maximiser la valeur renvoyée (le f1_score)
    study = optuna.create_study(direction="maximize")
    
    # On lance 50 essais (tu peux monter à 100 ou 200 si ça va vite)
    study.optimize(objective, n_trials=50)

    # ==========================================
    # 3. AFFICHAGE DU RÉSULTAT FINAL
    # ==========================================
    print("\n" + 50*"🌟")
    print("OPTIMISATION TERMINÉE")
    print(50*"🌟")
    print(f"Meilleur F1-Score atteint : {study.best_value * 100:.2f}%")
    print("Paramètres parfaits pour ce score :")
    for cle, valeur in study.best_params.items():
        print(f"  -> {cle} : {valeur}")

[I 2026-06-01 11:51:40,041] A new study created in memory with name: no-name-a4ee4887-9c7b-4074-a687-45b75c77ece9


🚀 Lancement de l'optimisation Bayésienne avec Optuna...


[I 2026-06-01 11:51:45,878] Trial 0 finished with value: 0.8235443359127235 and parameters: {'taille_fenetre': 12, 'facteur_std': 2.6840893414583618}. Best is trial 0 with value: 0.8235443359127235.
[I 2026-06-01 11:51:51,615] Trial 1 finished with value: 0.7708799623041583 and parameters: {'taille_fenetre': 120, 'facteur_std': 2.692285199850893}. Best is trial 0 with value: 0.8235443359127235.
[I 2026-06-01 11:51:56,991] Trial 2 finished with value: 0.6358169934640523 and parameters: {'taille_fenetre': 90, 'facteur_std': 1.7829294538794684}. Best is trial 0 with value: 0.8235443359127235.
[I 2026-06-01 11:52:02,369] Trial 3 finished with value: 0.3989674023081596 and parameters: {'taille_fenetre': 61, 'facteur_std': 1.1504493118033108}. Best is trial 0 with value: 0.8235443359127235.
[I 2026-06-01 11:52:08,467] Trial 4 finished with value: 0.46250825478777685 and parameters: {'taille_fenetre': 109, 'facteur_std': 1.3130412366358581}. Best is trial 0 with value: 0.8235443359127235.
[I 


🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟
OPTIMISATION TERMINÉE
🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟
Meilleur F1-Score atteint : 85.61%
Paramètres parfaits pour ce score :
  -> taille_fenetre : 16
  -> facteur_std : 2.0673180411932766


In [62]:
# ==========================================
# 1. LA FONCTION OBJECTIF POUR OPTUNA
# ==========================================
def objective(trial):
    """ 
    cette fonction suggère des val de param à tester -> donne le f1_score avec ce param 
    """
    # 1. Optuna suggère les paramètres à tester pour cet essai
    taille_fenetre_test = trial.suggest_int("taille_fenetre", 10, 150)
    facteur_std_test = trial.suggest_float("facteur_std", 1.0, 4.0)
    # num_bandes_test = trial.suggest_int("num_bandes", 1, 10)
    
    # 2. On prépare notre fonction de détection avec ces nouveaux paramètres
    # On utilise partial pour "geler" les paramètres sans exécuter la fonction
    methode_test = partial(
        local_threshold, 
        taille_fenetre=taille_fenetre_test,
        facteur_std=facteur_std_test,
        # num_bandes=num_bandes_test,
        metrique='Moyenne',
        visual_graph=False # Surtout pas de graphes pendant l'optimisation !
    )
    
    # 3. On appelle TA fonction d'évaluation (en mode silencieux)
    precision, recall, f1_score = evaluate_sequence_detection(
        methode_detection=methode_test,
        folder='train',
        img_type='VGA',
        sequence='sequence_2',
        dyn='low dyn with columns 3',
        update=False # <-- On cache les prints pour ne pas polluer le terminal
    )
    
    # 4. On renvoie uniquement le F1-score à Optuna car c'est ce qu'il doit maximiser
    return f1_score


# ==========================================
# 2. LANCEMENT DE L'OPTIMISATION
# ==========================================
if __name__ == "__main__":
    print("🚀 Lancement de l'optimisation Bayésienne avec Optuna...")
    
    # On crée l'étude en demandant de maximiser la valeur renvoyée (le f1_score)
    study = optuna.create_study(direction="maximize")
    
    # On lance 50 essais (tu peux monter à 100 ou 200 si ça va vite)
    study.optimize(objective, n_trials=50)

    # ==========================================
    # 3. AFFICHAGE DU RÉSULTAT FINAL
    # ==========================================
    print("\n" + 50*"🌟")
    print("OPTIMISATION TERMINÉE")
    print(50*"🌟")
    print(f"Meilleur F1-Score atteint : {study.best_value * 100:.2f}%")
    print("Paramètres parfaits pour ce score :")
    for cle, valeur in study.best_params.items():
        print(f"  -> {cle} : {valeur}")

[I 2026-06-01 13:13:21,899] A new study created in memory with name: no-name-42ef8aa9-85f6-4203-ab98-8c34fdf14631


🚀 Lancement de l'optimisation Bayésienne avec Optuna...


[I 2026-06-01 13:13:28,094] Trial 0 finished with value: 0.7781641168289292 and parameters: {'taille_fenetre': 92, 'facteur_std': 2.5399192153510937}. Best is trial 0 with value: 0.7781641168289292.
[I 2026-06-01 13:13:34,736] Trial 1 finished with value: 0.8088163662932295 and parameters: {'taille_fenetre': 69, 'facteur_std': 2.711137942798045}. Best is trial 1 with value: 0.8088163662932295.
[I 2026-06-01 13:13:42,224] Trial 2 finished with value: 0.6927705267055448 and parameters: {'taille_fenetre': 36, 'facteur_std': 1.6041953471923813}. Best is trial 1 with value: 0.8088163662932295.
[I 2026-06-01 13:13:49,194] Trial 3 finished with value: 0.6977777777777777 and parameters: {'taille_fenetre': 123, 'facteur_std': 2.2480367029928643}. Best is trial 1 with value: 0.8088163662932295.
[I 2026-06-01 13:13:55,328] Trial 4 finished with value: 0.7777193904361535 and parameters: {'taille_fenetre': 81, 'facteur_std': 3.4121812840383123}. Best is trial 1 with value: 0.8088163662932295.
[I 20


🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟
OPTIMISATION TERMINÉE
🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟
Meilleur F1-Score atteint : 85.31%
Paramètres parfaits pour ce score :
  -> taille_fenetre : 17
  -> facteur_std : 2.045505326211888


#### Comparaison avec LT_moy

In [53]:
print("Test de Local Thresholding avec la Métrique de la Moyenne")
evaluate_sequence_detection(
    methode_detection= partial (local_threshold, metrique='Moyenne'),
    folder="train",
    img_type="VGA",
    sequence="sequence_2",
    dyn="low dyn with columns 3", 
    update=True
)

Test de Local Thresholding avec la Métrique de la Moyenne

RÉSULTATS GLOBAUX - SÉQUENCE : type : VGA sequence : sequence_2 dyn : low dyn with columns 3
Total Vrais Positifs (VP) : 4033
Total Faux Négatifs (FN)  : 601 (Oublis)
Total Faux Positifs (FP)  : 17996 (Fausses alarmes)
--------------------------------------------------
Précision de la séquence  = 18.31%
Recall de la séquence     = 87.03%
F1_score de la séquence   = 30.25%



(0.18307685323891235, 0.8703064307293914, 0.3025165960319544)

### Optimisation des params pour local thresholding with band for metrique Moyenne LT_moy_band_opti

In [46]:
# ==========================================
# 1. LA FONCTION OBJECTIF POUR OPTUNA
# ==========================================
def objective(trial):
    """ 
    cette fonction suggère des val de param à tester -> donne le f1_score avec ce param 
    """
    # 1. Optuna suggère les paramètres à tester pour cet essai
    taille_fenetre_test = trial.suggest_int("taille_fenetre", 10, 150)
    facteur_std_test = trial.suggest_float("facteur_std", 1.0, 4.0)
    num_bandes_test = trial.suggest_int("num_bandes", 1, 10)
    
    # 2. On prépare notre fonction de détection avec ces nouveaux paramètres
    # On utilise partial pour "geler" les paramètres sans exécuter la fonction
    methode_test = partial(
        local_threshold_bandes, 
        taille_fenetre=taille_fenetre_test,
        facteur_std=facteur_std_test,
        num_bandes=num_bandes_test,
        metrique='Moyenne',
        visual_graph=False # Surtout pas de graphes pendant l'optimisation !
    )
    
    # 3. On appelle TA fonction d'évaluation (en mode silencieux)
    precision, recall, f1_score = evaluate_sequence_detection(
        methode_detection=methode_test,
        folder='train',
        img_type='VGA',
        sequence='sequence_2',
        dyn='low dyn with columns 3',
        update=False # <-- On cache les prints pour ne pas polluer le terminal
    )
    
    # 4. On renvoie uniquement le F1-score à Optuna car c'est ce qu'il doit maximiser
    return f1_score


# ==========================================
# 2. LANCEMENT DE L'OPTIMISATION
# ==========================================
if __name__ == "__main__":
    print("🚀 Lancement de l'optimisation Bayésienne avec Optuna...")
    
    # On crée l'étude en demandant de maximiser la valeur renvoyée (le f1_score)
    study = optuna.create_study(direction="maximize")
    
    # On lance 50 essais (tu peux monter à 100 ou 200 si ça va vite)
    study.optimize(objective, n_trials=50)

    # ==========================================
    # 3. AFFICHAGE DU RÉSULTAT FINAL
    # ==========================================
    print("\n" + 50*"🌟")
    print("OPTIMISATION TERMINÉE")
    print(50*"🌟")
    print(f"Meilleur F1-Score atteint : {study.best_value * 100:.2f}%")
    print("Paramètres parfaits pour ce score :")
    for cle, valeur in study.best_params.items():
        print(f"  -> {cle} : {valeur}")

[I 2026-05-29 16:15:56,882] A new study created in memory with name: no-name-53b52876-e893-44ed-80a1-933f4ced869e


🚀 Lancement de l'optimisation Bayésienne avec Optuna...


[I 2026-05-29 16:16:04,086] Trial 0 finished with value: 0.06489978035097924 and parameters: {'taille_fenetre': 126, 'facteur_std': 1.154386302122205, 'num_bandes': 3}. Best is trial 0 with value: 0.06489978035097924.
[I 2026-05-29 16:16:10,496] Trial 1 finished with value: 0.372669604377512 and parameters: {'taille_fenetre': 21, 'facteur_std': 2.5061703550552785, 'num_bandes': 9}. Best is trial 1 with value: 0.372669604377512.
[I 2026-05-29 16:16:16,688] Trial 2 finished with value: 0.4584742115940247 and parameters: {'taille_fenetre': 87, 'facteur_std': 2.951863085448963, 'num_bandes': 10}. Best is trial 2 with value: 0.4584742115940247.
[I 2026-05-29 16:16:24,791] Trial 3 finished with value: 0.0368146553713564 and parameters: {'taille_fenetre': 149, 'facteur_std': 1.0777191041643062, 'num_bandes': 4}. Best is trial 2 with value: 0.4584742115940247.
[I 2026-05-29 16:16:35,166] Trial 4 finished with value: 0.11607105362617058 and parameters: {'taille_fenetre': 79, 'facteur_std': 1.68


🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟
OPTIMISATION TERMINÉE
🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟
Meilleur F1-Score atteint : 85.45%
Paramètres parfaits pour ce score :
  -> taille_fenetre : 16
  -> facteur_std : 3.420181549013675
  -> num_bandes : 4


#### Comparaison avec LT_moy_band pas optimisée 

In [57]:
print("Test de Local Thresholding par bandes avec la Métrique Moyenne")
evaluate_sequence_detection(
    methode_detection= partial(local_threshold_bandes, metrique='Moyenne'),
    folder="train",
    img_type="VGA",
    sequence="sequence_2",
    dyn="low dyn with columns 3",
    update=True
)

Test de Local Thresholding par bandes avec la Métrique Moyenne

RÉSULTATS GLOBAUX - SÉQUENCE : type : VGA sequence : sequence_2 dyn : low dyn with columns 3
Total Vrais Positifs (VP) : 4424
Total Faux Négatifs (FN)  : 210 (Oublis)
Total Faux Positifs (FP)  : 246603 (Fausses alarmes)
--------------------------------------------------
Précision de la séquence  = 1.76%
Recall de la séquence     = 95.47%
F1_score de la séquence   = 3.46%



(0.017623602241989906, 0.9546827794561934, 0.03460832899816554)

### Optimisation des params pour CFAR_band RG in col 3 of seq 2

In [51]:
# ==========================================
# 1. LA FONCTION OBJECTIF POUR OPTUNA
# ==========================================
def objective(trial):
    """ 
    cette fonction suggère des val de param à tester -> donne le f1_score avec ce param 
    """
    # 1. Optuna suggère les paramètres à tester pour cet essai
    hauteur_bande = trial.suggest_int("hauteur_bande", 10, 150)
    train_cells = trial.suggest_int("train_cells", 4, 10)
    guard_cells = trial.suggest_int("guard_cells", 2, 5)
    multiplicateur_rupture = trial.suggest_float("multiplicateur_rupture", 2.0, 5.0)    

    # 2. On prépare notre fonction de détection avec ces nouveaux paramètres
    # On utilise partial pour "geler" les paramètres sans exécuter la fonction
    methode_test = partial(
        detecter_et_mesurer_defauts_complet, 
        hauteur_bande=hauteur_bande,
        train_cells=train_cells,
        guard_cells=guard_cells,
        multiplicateur_rupture=multiplicateur_rupture,
    )
    
    # 3. On appelle TA fonction d'évaluation (en mode silencieux)
    precision, recall, f1_score = evaluate_sequence_detection(
        methode_detection=methode_test,
        folder='train',
        img_type='VGA',
        sequence='sequence_2',
        dyn='low dyn with columns 3',
        update=False # <-- On cache les prints pour ne pas polluer le terminal
    )
    
    # 4. On renvoie uniquement le F1-score à Optuna car c'est ce qu'il doit maximiser
    return f1_score


# ==========================================
# 2. LANCEMENT DE L'OPTIMISATION
# ==========================================
if __name__ == "__main__":
    print("🚀 Lancement de l'optimisation Bayésienne avec Optuna...")
    
    # On crée l'étude en demandant de maximiser la valeur renvoyée (le f1_score)
    study = optuna.create_study(direction="maximize")
    
    # On lance 50 essais (tu peux monter à 100 ou 200 si ça va vite)
    study.optimize(objective, n_trials=50)

    # ==========================================
    # 3. AFFICHAGE DU RÉSULTAT FINAL
    # ==========================================
    print("\n" + 50*"🌟")
    print("OPTIMISATION TERMINÉE")
    print(50*"🌟")
    print(f"Meilleur F1-Score atteint : {study.best_value * 100:.2f}%")
    print("Paramètres parfaits pour ce score :")
    for cle, valeur in study.best_params.items():
        print(f"  -> {cle} : {valeur}")

[I 2026-06-01 11:15:03,453] A new study created in memory with name: no-name-d0e73445-e9cc-4532-855f-8e5309c1e4d6


🚀 Lancement de l'optimisation Bayésienne avec Optuna...


[I 2026-06-01 11:15:25,428] Trial 0 finished with value: 0.14706522384221252 and parameters: {'hauteur_bande': 19, 'train_cells': 4, 'guard_cells': 5, 'multiplicateur_rupture': 3.0489202659743793}. Best is trial 0 with value: 0.14706522384221252.
[I 2026-06-01 11:15:35,672] Trial 1 finished with value: 0.7623503808487486 and parameters: {'hauteur_bande': 150, 'train_cells': 5, 'guard_cells': 5, 'multiplicateur_rupture': 3.3400709913594255}. Best is trial 1 with value: 0.7623503808487486.
[I 2026-06-01 11:15:49,916] Trial 2 finished with value: 0.629995627459554 and parameters: {'hauteur_bande': 70, 'train_cells': 9, 'guard_cells': 2, 'multiplicateur_rupture': 4.435057392393215}. Best is trial 1 with value: 0.7623503808487486.
[I 2026-06-01 11:16:04,615] Trial 3 finished with value: 0.5747018204645323 and parameters: {'hauteur_bande': 59, 'train_cells': 7, 'guard_cells': 3, 'multiplicateur_rupture': 4.640232740709345}. Best is trial 1 with value: 0.7623503808487486.
[I 2026-06-01 11:16:


🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟
OPTIMISATION TERMINÉE
🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟
Meilleur F1-Score atteint : 81.79%
Paramètres parfaits pour ce score :
  -> hauteur_bande : 134
  -> train_cells : 9
  -> guard_cells : 2
  -> multiplicateur_rupture : 4.349861895099086


#### Comparaison CFAR_band 

In [61]:
print("Test de CFAR_band et Region Growing")
evaluate_sequence_detection(
    methode_detection= detecter_et_mesurer_defauts_complet,
    folder="train",
    img_type="VGA",
    sequence="sequence_2",
    dyn="low dyn with columns 3", 
    update = True
)

Test de CFAR_band et Region Growing

RÉSULTATS GLOBAUX - SÉQUENCE : type : VGA sequence : sequence_2 dyn : low dyn with columns 3
Total Vrais Positifs (VP) : 3636
Total Faux Négatifs (FN)  : 998 (Oublis)
Total Faux Positifs (FP)  : 8526 (Fausses alarmes)
--------------------------------------------------
Précision de la séquence  = 29.90%
Recall de la séquence     = 78.46%
F1_score de la séquence   = 43.30%



(0.2989639861864825, 0.7846353042727665, 0.4329602286258633)

# TEST CFAR sur col 1 (juste full col)

# TEST avec HD

### LT_band_opti

In [68]:
# ==========================================
# 1. LA FONCTION OBJECTIF POUR OPTUNA
# ==========================================
def objective(trial):
    """ 
    cette fonction suggère des val de param à tester -> donne le f1_score avec ce param 
    """
    # 1. Optuna suggère les paramètres à tester pour cet essai
    taille_fenetre_test = trial.suggest_int("taille_fenetre", 10, 150)
    facteur_std_test = trial.suggest_float("facteur_std", 1.0, 4.0)
    num_bandes_test = trial.suggest_int("num_bandes", 1, 10)
    
    # 2. On prépare notre fonction de détection avec ces nouveaux paramètres
    # On utilise partial pour "geler" les paramètres sans exécuter la fonction
    methode_test = partial(
        local_threshold_bandes, 
        taille_fenetre=taille_fenetre_test,
        facteur_std=facteur_std_test,
        num_bandes=num_bandes_test,
        metrique='Moyenne',
        visual_graph=False # Surtout pas de graphes pendant l'optimisation !
    )
    
    # 3. On appelle TA fonction d'évaluation (en mode silencieux)
    precision, recall, f1_score = evaluate_sequence_detection(
        methode_detection=methode_test,
        folder='train',
        img_type='HD',
        sequence='sequence_2',
        dyn='low dyn with columns 3',
        update=False # <-- On cache les prints pour ne pas polluer le terminal
    )
    
    # 4. On renvoie uniquement le F1-score à Optuna car c'est ce qu'il doit maximiser
    return f1_score


# ==========================================
# 2. LANCEMENT DE L'OPTIMISATION
# ==========================================
if __name__ == "__main__":
    print("🚀 Lancement de l'optimisation Bayésienne avec Optuna...")
    
    # On crée l'étude en demandant de maximiser la valeur renvoyée (le f1_score)
    study = optuna.create_study(direction="maximize")
    
    # On lance 50 essais (tu peux monter à 100 ou 200 si ça va vite)
    study.optimize(objective, n_trials=50)

    # ==========================================
    # 3. AFFICHAGE DU RÉSULTAT FINAL
    # ==========================================
    print("\n" + 50*"🌟")
    print("OPTIMISATION TERMINÉE")
    print(50*"🌟")
    print(f"Meilleur F1-Score atteint : {study.best_value * 100:.2f}%")
    print("Paramètres parfaits pour ce score :")
    for cle, valeur in study.best_params.items():
        print(f"  -> {cle} : {valeur}")

[I 2026-06-01 13:58:18,180] A new study created in memory with name: no-name-1a5f2d75-7d73-4650-a289-1964e1edde14


🚀 Lancement de l'optimisation Bayésienne avec Optuna...


[I 2026-06-01 13:58:23,679] Trial 0 finished with value: 0.40239867659222495 and parameters: {'taille_fenetre': 115, 'facteur_std': 1.8059709628137104, 'num_bandes': 2}. Best is trial 0 with value: 0.40239867659222495.
[I 2026-06-01 13:58:28,636] Trial 1 finished with value: 0.3977218794494542 and parameters: {'taille_fenetre': 136, 'facteur_std': 3.6347548244004813, 'num_bandes': 6}. Best is trial 0 with value: 0.40239867659222495.
[I 2026-06-01 13:58:34,199] Trial 2 finished with value: 0.23460752845053987 and parameters: {'taille_fenetre': 130, 'facteur_std': 1.9162269937826717, 'num_bandes': 9}. Best is trial 0 with value: 0.40239867659222495.
[I 2026-06-01 13:58:38,992] Trial 3 finished with value: 0.43807158179970646 and parameters: {'taille_fenetre': 11, 'facteur_std': 1.2397069013553814, 'num_bandes': 6}. Best is trial 3 with value: 0.43807158179970646.
[I 2026-06-01 13:58:43,627] Trial 4 finished with value: 0.3629014477842288 and parameters: {'taille_fenetre': 54, 'facteur_st


🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟
OPTIMISATION TERMINÉE
🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟
Meilleur F1-Score atteint : 59.25%
Paramètres parfaits pour ce score :
  -> taille_fenetre : 54
  -> facteur_std : 2.067518715814121
  -> num_bandes : 4


#### Comparaison avec LT_moy_band 

In [69]:
print("Test de Local Thresholding par bandes avec la Métrique Moyenne")
evaluate_sequence_detection(
    methode_detection= partial(local_threshold_bandes, metrique='Moyenne'),
    folder="train",
    img_type="HD",
    sequence="sequence_2",
    dyn="low dyn with columns 3",
    update=True
)

Test de Local Thresholding par bandes avec la Métrique Moyenne

RÉSULTATS GLOBAUX - SÉQUENCE : type : HD sequence : sequence_2 dyn : low dyn with columns 3
Total Vrais Positifs (VP) : 3818
Total Faux Négatifs (FN)  : 1246 (Oublis)
Total Faux Positifs (FP)  : 21372 (Fausses alarmes)
--------------------------------------------------
Précision de la séquence  = 15.16%
Recall de la séquence     = 75.39%
F1_score de la séquence   = 25.24%



(0.1515680825724494, 0.7539494470774092, 0.25239637733853376)

# TEST avec SXGA
### LT_moy_band_opti

In [70]:
# ==========================================
# 1. LA FONCTION OBJECTIF POUR OPTUNA
# ==========================================
def objective(trial):
    """ 
    cette fonction suggère des val de param à tester -> donne le f1_score avec ce param 
    """
    # 1. Optuna suggère les paramètres à tester pour cet essai
    taille_fenetre_test = trial.suggest_int("taille_fenetre", 10, 150)
    facteur_std_test = trial.suggest_float("facteur_std", 1.0, 4.0)
    num_bandes_test = trial.suggest_int("num_bandes", 1, 10)
    
    # 2. On prépare notre fonction de détection avec ces nouveaux paramètres
    # On utilise partial pour "geler" les paramètres sans exécuter la fonction
    methode_test = partial(
        local_threshold_bandes, 
        taille_fenetre=taille_fenetre_test,
        facteur_std=facteur_std_test,
        num_bandes=num_bandes_test,
        metrique='Moyenne',
        visual_graph=False # Surtout pas de graphes pendant l'optimisation !
    )
    
    # 3. On appelle TA fonction d'évaluation (en mode silencieux)
    precision, recall, f1_score = evaluate_sequence_detection(
        methode_detection=methode_test,
        folder='train',
        img_type='SXGA',
        sequence='sequence_2',
        dyn='low dyn with columns 3',
        update=False # <-- On cache les prints pour ne pas polluer le terminal
    )
    
    # 4. On renvoie uniquement le F1-score à Optuna car c'est ce qu'il doit maximiser
    return f1_score


# ==========================================
# 2. LANCEMENT DE L'OPTIMISATION
# ==========================================
if __name__ == "__main__":
    print("🚀 Lancement de l'optimisation Bayésienne avec Optuna...")
    
    # On crée l'étude en demandant de maximiser la valeur renvoyée (le f1_score)
    study = optuna.create_study(direction="maximize")
    
    # On lance 50 essais (tu peux monter à 100 ou 200 si ça va vite)
    study.optimize(objective, n_trials=50)

    # ==========================================
    # 3. AFFICHAGE DU RÉSULTAT FINAL
    # ==========================================
    print("\n" + 50*"🌟")
    print("OPTIMISATION TERMINÉE")
    print(50*"🌟")
    print(f"Meilleur F1-Score atteint : {study.best_value * 100:.2f}%")
    print("Paramètres parfaits pour ce score :")
    for cle, valeur in study.best_params.items():
        print(f"  -> {cle} : {valeur}")

[I 2026-06-01 14:06:00,911] A new study created in memory with name: no-name-08c35c0a-cffd-4bc0-8d36-2f88408eedca


🚀 Lancement de l'optimisation Bayésienne avec Optuna...


[I 2026-06-01 14:06:18,323] Trial 0 finished with value: 0.2619976638417377 and parameters: {'taille_fenetre': 37, 'facteur_std': 1.4056754476989166, 'num_bandes': 7}. Best is trial 0 with value: 0.2619976638417377.
[I 2026-06-01 14:06:34,995] Trial 1 finished with value: 0.18786721776360907 and parameters: {'taille_fenetre': 73, 'facteur_std': 1.6271999842452356, 'num_bandes': 9}. Best is trial 0 with value: 0.2619976638417377.
[I 2026-06-01 14:06:49,101] Trial 2 finished with value: 0.04230876287355449 and parameters: {'taille_fenetre': 134, 'facteur_std': 1.0252843685837871, 'num_bandes': 10}. Best is trial 0 with value: 0.2619976638417377.
[I 2026-06-01 14:07:01,748] Trial 3 finished with value: 0.4637250742469241 and parameters: {'taille_fenetre': 45, 'facteur_std': 2.3089495889544343, 'num_bandes': 4}. Best is trial 3 with value: 0.4637250742469241.
[I 2026-06-01 14:07:14,552] Trial 4 finished with value: 0.3073273046109411 and parameters: {'taille_fenetre': 137, 'facteur_std': 2


🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟
OPTIMISATION TERMINÉE
🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟
Meilleur F1-Score atteint : 70.67%
Paramètres parfaits pour ce score :
  -> taille_fenetre : 15
  -> facteur_std : 1.9082320894348743
  -> num_bandes : 5


#### Compa avec LT_moy

In [71]:
print("Test de Local Thresholding par bandes avec la Métrique Moyenne")
evaluate_sequence_detection(
    methode_detection= partial(local_threshold_bandes, metrique='Moyenne'),
    folder="train",
    img_type="SXGA",
    sequence="sequence_2",
    dyn="low dyn with columns 3",
    update=True
)

Test de Local Thresholding par bandes avec la Métrique Moyenne

RÉSULTATS GLOBAUX - SÉQUENCE : type : SXGA sequence : sequence_2 dyn : low dyn with columns 3
Total Vrais Positifs (VP) : 5467
Total Faux Négatifs (FN)  : 565 (Oublis)
Total Faux Positifs (FP)  : 65239 (Fausses alarmes)
--------------------------------------------------
Précision de la séquence  = 7.73%
Recall de la séquence     = 90.63%
F1_score de la séquence   = 14.25%



(0.07732017084830142, 0.9063328912466844, 0.14248481847324665)

# TEST sur toutes les séquences d'un type 
## VGA 

### LT_band_moy 

In [76]:
print("Test de Local Thresholding avec la Métrique de la Moyenne")
evaluate_type_detection(
    methode_detection= partial (local_threshold_bandes, metrique='Moyenne'),
    folder="train",
    img_type="VGA",
    update=True
)

Test de Local Thresholding avec la Métrique de la Moyenne

RÉSULTATS GLOBAUX - TYPE : VGA
Séquences : ['sequence_1', 'sequence_2', 'sequence_3']
Dynamiques : ['low dyn with columns 1', 'low dyn with columns 2', 'low dyn with columns 3']
Total Vrais Positifs (VP) : 21665
Total Faux Négatifs (FN)  : 2957 (Oublis)
Total Faux Positifs (FP)  : 850405 (Fausses alarmes)
------------------------------------------------------------
Précision globale         = 2.48%
Recall global             = 87.99%
F1_score global           = 4.83%



(0.024843189193528042, 0.8799041507594834, 0.048322054841573245)

### LT_moy_band_opti

In [ ]:
# ==========================================
# 1. LA FONCTION OBJECTIF POUR OPTUNA
# ==========================================
def objective(trial):
    """ 
    cette fonction suggère des val de param à tester -> donne le f1_score avec ce param 
    """
    # 1. Optuna suggère les paramètres à tester pour cet essai
    taille_fenetre_test = trial.suggest_int("taille_fenetre", 10, 150)
    facteur_std_test = trial.suggest_float("facteur_std", 1.0, 4.0)
    num_bandes_test = trial.suggest_int("num_bandes", 1, 10)
    
    # 2. On prépare notre fonction de détection avec ces nouveaux paramètres
    # On utilise partial pour "geler" les paramètres sans exécuter la fonction
    methode_test = partial(
        local_threshold_bandes, 
        taille_fenetre=taille_fenetre_test,
        facteur_std=facteur_std_test,
        num_bandes=num_bandes_test,
        metrique='Moyenne',
        visual_graph=False # Surtout pas de graphes pendant l'optimisation !
    )
    
    # 3. On appelle LA NOUVELLE fonction d'évaluation ULTIME (en mode silencieux)
    precision, recall, f1_score = evaluate_type_detection(
        methode_detection=methode_test,
        folder='train',
        img_type='VGA',
        sequences=['sequence_1', 'sequence_2', 'sequence_3'], 
        dyns=['low dyn with columns 1', 'low dyn with columns 2', 'low dyn with columns 3'],
        update=False # <-- On cache les prints pour ne pas polluer le terminal
    )
    
    # 4. On renvoie uniquement le F1-score à Optuna car c'est ce qu'il doit maximiser
    return f1_score


# ==========================================
# 2. LANCEMENT DE L'OPTIMISATION
# ==========================================
if __name__ == "__main__":
    print("🚀 Lancement de l'optimisation Bayésienne TOTALE avec Optuna...")
    
    # On crée l'étude en demandant de maximiser la valeur renvoyée (le f1_score)
    study = optuna.create_study(direction="maximize")
    
    # On lance 50 essais avec la barre de progression activée
    study.optimize(objective, n_trials=50, show_progress_bar=True)

    # ==========================================
    # 3. AFFICHAGE DU RÉSULTAT FINAL
    # ==========================================
    print("\n" + 50*"🌟")
    print("OPTIMISATION TERMINÉE")
    print(50*"🌟")
    print(f"Meilleur F1-Score GLOBAL atteint : {study.best_value * 100:.2f}%")
    print("Paramètres parfaits pour ce score :")
    for cle, valeur in study.best_params.items():
        print(f"  -> {cle} : {valeur}")

[I 2026-06-01 14:39:58,779] A new study created in memory with name: no-name-f8f6d63b-5858-4aaa-a34e-2a89690dd9e3


🚀 Lancement de l'optimisation Bayésienne TOTALE avec Optuna...


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-06-01 14:40:37,769] Trial 0 finished with value: 0.3763557015080154 and parameters: {'taille_fenetre': 81, 'facteur_std': 2.3574334965455965, 'num_bandes': 4}. Best is trial 0 with value: 0.3763557015080154.
[I 2026-06-01 14:41:17,058] Trial 1 finished with value: 0.45764829142488717 and parameters: {'taille_fenetre': 126, 'facteur_std': 3.883806384912953, 'num_bandes': 5}. Best is trial 1 with value: 0.45764829142488717.
[I 2026-06-01 14:41:53,719] Trial 2 finished with value: 0.5458454386984312 and parameters: {'taille_fenetre': 134, 'facteur_std': 3.9073725768277177, 'num_bandes': 2}. Best is trial 2 with value: 0.5458454386984312.
[I 2026-06-01 14:42:30,837] Trial 3 finished with value: 0.4230206045392741 and parameters: {'taille_fenetre': 105, 'facteur_std': 2.6362797453366573, 'num_bandes': 4}. Best is trial 2 with value: 0.5458454386984312.
[I 2026-06-01 14:43:11,770] Trial 4 finished with value: 0.24769270152652215 and parameters: {'taille_fenetre': 100, 'facteur_std': 

### LT_moy_VGA

In [81]:
print("Test de Local Thresholding avec la Métrique de la Moyenne")
evaluate_type_detection(
    methode_detection= partial (local_threshold, metrique='Moyenne'),
    folder="train",
    img_type="VGA",
    update=True
)

Test de Local Thresholding avec la Métrique de la Moyenne

RÉSULTATS GLOBAUX - TYPE : VGA
Séquences : ['sequence_1', 'sequence_2', 'sequence_3']
Dynamiques : ['low dyn with columns 1', 'low dyn with columns 2', 'low dyn with columns 3']
Total Vrais Positifs (VP) : 18996
Total Faux Négatifs (FN)  : 5626 (Oublis)
Total Faux Positifs (FP)  : 61157 (Fausses alarmes)
------------------------------------------------------------
Précision globale         = 23.70%
Recall global             = 77.15%
F1_score global           = 36.26%



(0.23699674372762092, 0.7715051579887905, 0.362605583392985)

### LT_moy_opti_VGA

In [86]:
# ==========================================
# 1. LA FONCTION OBJECTIF POUR OPTUNA
# ==========================================
def objective(trial):
    """ 
    cette fonction suggère des val de param à tester -> donne le f1_score avec ce param 
    """
    # 1. Optuna suggère les paramètres à tester pour cet essai
    taille_fenetre_test = trial.suggest_int("taille_fenetre", 10, 150)
    facteur_std_test = trial.suggest_float("facteur_std", 1.0, 4.0)
    # num_bandes_test = trial.suggest_int("num_bandes", 1, 10)
    
    # 2. On prépare notre fonction de détection avec ces nouveaux paramètres
    # On utilise partial pour "geler" les paramètres sans exécuter la fonction
    methode_test = partial(
        local_threshold, 
        taille_fenetre=taille_fenetre_test,
        facteur_std=facteur_std_test,
        # num_bandes=num_bandes_test,
        metrique='Moyenne',
        visual_graph=False # Surtout pas de graphes pendant l'optimisation !
    )
    
    # 3. On appelle LA NOUVELLE fonction d'évaluation ULTIME (en mode silencieux)
    precision, recall, f1_score = evaluate_type_detection(
        methode_detection=methode_test,
        folder='train',
        img_type='VGA',
        sequences=['sequence_1', 'sequence_2', 'sequence_3'], 
        dyns=['low dyn with columns 1', 'low dyn with columns 2', 'low dyn with columns 3'],
        update=False # <-- On cache les prints pour ne pas polluer le terminal
    )
    
    # 4. On renvoie uniquement le F1-score à Optuna car c'est ce qu'il doit maximiser
    return f1_score


# ==========================================
# 2. LANCEMENT DE L'OPTIMISATION
# ==========================================
if __name__ == "__main__":
    print("🚀 Lancement de l'optimisation Bayésienne TOTALE avec Optuna...")
    
    # On crée l'étude en demandant de maximiser la valeur renvoyée (le f1_score)
    study = optuna.create_study(direction="maximize")
    
    # On lance 50 essais avec la barre de progression activée
    study.optimize(objective, n_trials=50, show_progress_bar=True)

    # ==========================================
    # 3. AFFICHAGE DU RÉSULTAT FINAL
    # ==========================================
    print("\n" + 50*"🌟")
    print("OPTIMISATION TERMINÉE")
    print(50*"🌟")
    print(f"Meilleur F1-Score GLOBAL atteint : {study.best_value * 100:.2f}%")
    print("Paramètres parfaits pour ce score :")
    for cle, valeur in study.best_params.items():
        print(f"  -> {cle} : {valeur}")

[I 2026-06-01 22:12:24,887] A new study created in memory with name: no-name-effb6a98-88fd-4080-9ee9-56423444c9cf


🚀 Lancement de l'optimisation Bayésienne TOTALE avec Optuna...


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-06-01 22:13:06,625] Trial 0 finished with value: 0.5302492165348456 and parameters: {'taille_fenetre': 27, 'facteur_std': 3.567296504329141}. Best is trial 0 with value: 0.5302492165348456.
[I 2026-06-01 22:13:47,444] Trial 1 finished with value: 0.5741959376691451 and parameters: {'taille_fenetre': 112, 'facteur_std': 3.2400722073099355}. Best is trial 1 with value: 0.5741959376691451.
[I 2026-06-01 22:14:24,875] Trial 2 finished with value: 0.7126377126141216 and parameters: {'taille_fenetre': 10, 'facteur_std': 1.7526562322131571}. Best is trial 2 with value: 0.7126377126141216.
[I 2026-06-01 22:15:02,328] Trial 3 finished with value: 0.6296409941699908 and parameters: {'taille_fenetre': 91, 'facteur_std': 2.0915998465099115}. Best is trial 2 with value: 0.7126377126141216.
[I 2026-06-01 22:15:51,624] Trial 4 finished with value: 0.5321060842433697 and parameters: {'taille_fenetre': 103, 'facteur_std': 1.4001765717196575}. Best is trial 2 with value: 0.7126377126141216.
[I 2

# SXGA

In [80]:
# ==========================================
# 1. LA FONCTION OBJECTIF POUR OPTUNA
# ==========================================
def objective(trial):
    """ 
    cette fonction suggère des val de param à tester -> donne le f1_score avec ce param 
    """
    # 1. Optuna suggère les paramètres à tester pour cet essai
    taille_fenetre_test = trial.suggest_int("taille_fenetre", 10, 150)
    facteur_std_test = trial.suggest_float("facteur_std", 1.0, 4.0)
    num_bandes_test = trial.suggest_int("num_bandes", 1, 10)
    
    # 2. On prépare notre fonction de détection avec ces nouveaux paramètres
    # On utilise partial pour "geler" les paramètres sans exécuter la fonction
    methode_test = partial(
        local_threshold_bandes, 
        taille_fenetre=taille_fenetre_test,
        facteur_std=facteur_std_test,
        num_bandes=num_bandes_test,
        metrique='Moyenne',
        visual_graph=False # Surtout pas de graphes pendant l'optimisation !
    )
    
    # 3. On appelle LA NOUVELLE fonction d'évaluation ULTIME (en mode silencieux)
    precision, recall, f1_score = evaluate_type_detection(
        methode_detection=methode_test,
        folder='train',
        img_type='SXGA',
        sequences=['sequence_1', 'sequence_2', 'sequence_3'], 
        dyns=['low dyn with columns 1', 'low dyn with columns 2', 'low dyn with columns 3'],
        update=False # <-- On cache les prints pour ne pas polluer le terminal
    )
    
    # 4. On renvoie uniquement le F1-score à Optuna car c'est ce qu'il doit maximiser
    return f1_score


# ==========================================
# 2. LANCEMENT DE L'OPTIMISATION
# ==========================================
if __name__ == "__main__":
    print("🚀 Lancement de l'optimisation Bayésienne TOTALE avec Optuna...")
    
    # On crée l'étude en demandant de maximiser la valeur renvoyée (le f1_score)
    study = optuna.create_study(direction="maximize")
    
    # On lance 50 essais avec la barre de progression activée
    study.optimize(objective, n_trials=50, show_progress_bar=True)

    # ==========================================
    # 3. AFFICHAGE DU RÉSULTAT FINAL
    # ==========================================
    print("\n" + 50*"🌟")
    print("OPTIMISATION TERMINÉE")
    print(50*"🌟")
    print(f"Meilleur F1-Score GLOBAL atteint : {study.best_value * 100:.2f}%")
    print("Paramètres parfaits pour ce score :")
    for cle, valeur in study.best_params.items():
        print(f"  -> {cle} : {valeur}")

[I 2026-06-01 16:24:36,013] A new study created in memory with name: no-name-35cf731b-f3bd-4cc4-885f-20e523c96b4c


🚀 Lancement de l'optimisation Bayésienne TOTALE avec Optuna...


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-06-01 16:26:32,016] Trial 0 finished with value: 0.45220063306158076 and parameters: {'taille_fenetre': 82, 'facteur_std': 1.6473999385826432, 'num_bandes': 3}. Best is trial 0 with value: 0.45220063306158076.
[I 2026-06-01 16:28:16,089] Trial 1 finished with value: 0.5121726157109964 and parameters: {'taille_fenetre': 133, 'facteur_std': 3.7108222951287475, 'num_bandes': 2}. Best is trial 1 with value: 0.5121726157109964.
[I 2026-06-01 16:30:11,603] Trial 2 finished with value: 0.5742109058204176 and parameters: {'taille_fenetre': 78, 'facteur_std': 3.86811976858698, 'num_bandes': 4}. Best is trial 2 with value: 0.5742109058204176.
[I 2026-06-01 16:32:00,081] Trial 3 finished with value: 0.5412263037289078 and parameters: {'taille_fenetre': 110, 'facteur_std': 3.8595170783157196, 'num_bandes': 8}. Best is trial 2 with value: 0.5742109058204176.
[I 2026-06-01 16:33:46,461] Trial 4 finished with value: 0.49740858047797293 and parameters: {'taille_fenetre': 55, 'facteur_std': 1.4

### LT_moy_SXGA

In [82]:
print("Test de Local Thresholding avec la Métrique de la Moyenne")
evaluate_type_detection(
    methode_detection= partial (local_threshold, metrique='Moyenne'),
    folder="train",
    img_type="SXGA",
    update=True
)

Test de Local Thresholding avec la Métrique de la Moyenne

RÉSULTATS GLOBAUX - TYPE : SXGA
Séquences : ['sequence_1', 'sequence_2', 'sequence_3']
Dynamiques : ['low dyn with columns 1', 'low dyn with columns 2', 'low dyn with columns 3']
Total Vrais Positifs (VP) : 51256
Total Faux Négatifs (FN)  : 23228 (Oublis)
Total Faux Positifs (FP)  : 118900 (Fausses alarmes)
------------------------------------------------------------
Précision globale         = 30.12%
Recall global             = 68.81%
F1_score global           = 41.90%



(0.3012294600249183, 0.6881477901294237, 0.419032047089601)

### LT_moy_opti sur tous les seq de SXGA

In [83]:
# ==========================================
# 1. LA FONCTION OBJECTIF POUR OPTUNA
# ==========================================
def objective(trial):
    """ 
    cette fonction suggère des val de param à tester -> donne le f1_score avec ce param 
    """
    # 1. Optuna suggère les paramètres à tester pour cet essai
    taille_fenetre_test = trial.suggest_int("taille_fenetre", 10, 150)
    facteur_std_test = trial.suggest_float("facteur_std", 1.0, 4.0)
    # num_bandes_test = trial.suggest_int("num_bandes", 1, 10)
    
    # 2. On prépare notre fonction de détection avec ces nouveaux paramètres
    # On utilise partial pour "geler" les paramètres sans exécuter la fonction
    methode_test = partial(
        local_threshold, 
        taille_fenetre=taille_fenetre_test,
        facteur_std=facteur_std_test,
        # num_bandes=num_bandes_test,
        metrique='Moyenne',
        visual_graph=False # Surtout pas de graphes pendant l'optimisation !
    )
    
    # 3. On appelle LA NOUVELLE fonction d'évaluation ULTIME (en mode silencieux)
    precision, recall, f1_score = evaluate_type_detection(
        methode_detection=methode_test,
        folder='train',
        img_type='SXGA',
        sequences=['sequence_1', 'sequence_2', 'sequence_3'], 
        dyns=['low dyn with columns 1', 'low dyn with columns 2', 'low dyn with columns 3'],
        update=False # <-- On cache les prints pour ne pas polluer le terminal
    )
    
    # 4. On renvoie uniquement le F1-score à Optuna car c'est ce qu'il doit maximiser
    return f1_score


# ==========================================
# 2. LANCEMENT DE L'OPTIMISATION
# ==========================================
if __name__ == "__main__":
    print("🚀 Lancement de l'optimisation Bayésienne TOTALE avec Optuna...")
    
    # On crée l'étude en demandant de maximiser la valeur renvoyée (le f1_score)
    study = optuna.create_study(direction="maximize")
    
    # On lance 50 essais avec la barre de progression activée
    study.optimize(objective, n_trials=50, show_progress_bar=True)

    # ==========================================
    # 3. AFFICHAGE DU RÉSULTAT FINAL
    # ==========================================
    print("\n" + 50*"🌟")
    print("OPTIMISATION TERMINÉE")
    print(50*"🌟")
    print(f"Meilleur F1-Score GLOBAL atteint : {study.best_value * 100:.2f}%")
    print("Paramètres parfaits pour ce score :")
    for cle, valeur in study.best_params.items():
        print(f"  -> {cle} : {valeur}")

[I 2026-06-01 18:03:53,824] A new study created in memory with name: no-name-007bd0e5-db6e-4eb7-b431-37e7707e5dfc


🚀 Lancement de l'optimisation Bayésienne TOTALE avec Optuna...


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-06-01 18:05:37,928] Trial 0 finished with value: 0.4643949719824322 and parameters: {'taille_fenetre': 36, 'facteur_std': 3.5348766323883756}. Best is trial 0 with value: 0.4643949719824322.
[I 2026-06-01 18:07:21,559] Trial 1 finished with value: 0.48956137028010266 and parameters: {'taille_fenetre': 146, 'facteur_std': 2.303751516643783}. Best is trial 1 with value: 0.48956137028010266.
[I 2026-06-01 18:09:12,885] Trial 2 finished with value: 0.5535152397501527 and parameters: {'taille_fenetre': 31, 'facteur_std': 2.6254956078349787}. Best is trial 2 with value: 0.5535152397501527.
[I 2026-06-01 18:11:05,375] Trial 3 finished with value: 0.46177319507478204 and parameters: {'taille_fenetre': 89, 'facteur_std': 3.6924516439606334}. Best is trial 2 with value: 0.5535152397501527.
[I 2026-06-01 18:12:48,421] Trial 4 finished with value: 0.49899211034746777 and parameters: {'taille_fenetre': 100, 'facteur_std': 1.6657787094971632}. Best is trial 2 with value: 0.5535152397501527.


### LT_moy_opti sur toutes les seq de HD

In [84]:
# ==========================================
# 1. LA FONCTION OBJECTIF POUR OPTUNA
# ==========================================
def objective(trial):
    """ 
    cette fonction suggère des val de param à tester -> donne le f1_score avec ce param 
    """
    # 1. Optuna suggère les paramètres à tester pour cet essai
    taille_fenetre_test = trial.suggest_int("taille_fenetre", 10, 150)
    facteur_std_test = trial.suggest_float("facteur_std", 1.0, 4.0)
    # num_bandes_test = trial.suggest_int("num_bandes", 1, 10)
    
    # 2. On prépare notre fonction de détection avec ces nouveaux paramètres
    # On utilise partial pour "geler" les paramètres sans exécuter la fonction
    methode_test = partial(
        local_threshold, 
        taille_fenetre=taille_fenetre_test,
        facteur_std=facteur_std_test,
        # num_bandes=num_bandes_test,
        metrique='Moyenne',
        visual_graph=False # Surtout pas de graphes pendant l'optimisation !
    )
    
    # 3. On appelle LA NOUVELLE fonction d'évaluation ULTIME (en mode silencieux)
    precision, recall, f1_score = evaluate_type_detection(
        methode_detection=methode_test,
        folder='train',
        img_type='HD',
        sequences=['sequence_1', 'sequence_2', 'sequence_3'], 
        dyns=['low dyn with columns 1', 'low dyn with columns 2', 'low dyn with columns 3'],
        update=False # <-- On cache les prints pour ne pas polluer le terminal
    )
    
    # 4. On renvoie uniquement le F1-score à Optuna car c'est ce qu'il doit maximiser
    return f1_score


# ==========================================
# 2. LANCEMENT DE L'OPTIMISATION
# ==========================================
if __name__ == "__main__":
    print("🚀 Lancement de l'optimisation Bayésienne TOTALE avec Optuna...")
    
    # On crée l'étude en demandant de maximiser la valeur renvoyée (le f1_score)
    study = optuna.create_study(direction="maximize")
    
    # On lance 50 essais avec la barre de progression activée
    study.optimize(objective, n_trials=50, show_progress_bar=True)

    # ==========================================
    # 3. AFFICHAGE DU RÉSULTAT FINAL
    # ==========================================
    print("\n" + 50*"🌟")
    print("OPTIMISATION TERMINÉE")
    print(50*"🌟")
    print(f"Meilleur F1-Score GLOBAL atteint : {study.best_value * 100:.2f}%")
    print("Paramètres parfaits pour ce score :")
    for cle, valeur in study.best_params.items():
        print(f"  -> {cle} : {valeur}")

[I 2026-06-01 19:51:41,000] A new study created in memory with name: no-name-986523df-d6e7-4ddb-a209-3eb4429fc178


🚀 Lancement de l'optimisation Bayésienne TOTALE avec Optuna...


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-06-01 19:52:22,655] Trial 0 finished with value: 0.3176208493935429 and parameters: {'taille_fenetre': 47, 'facteur_std': 3.306422749575178}. Best is trial 0 with value: 0.3176208493935429.
[I 2026-06-01 19:53:00,869] Trial 1 finished with value: 0.2849598534245837 and parameters: {'taille_fenetre': 79, 'facteur_std': 3.698289344628675}. Best is trial 0 with value: 0.3176208493935429.
[I 2026-06-01 19:53:39,084] Trial 2 finished with value: 0.3139417850960184 and parameters: {'taille_fenetre': 18, 'facteur_std': 3.0939660144387062}. Best is trial 0 with value: 0.3176208493935429.
[I 2026-06-01 19:54:18,595] Trial 3 finished with value: 0.3639025513645112 and parameters: {'taille_fenetre': 80, 'facteur_std': 2.9598777476948883}. Best is trial 3 with value: 0.3639025513645112.
[I 2026-06-01 19:54:57,750] Trial 4 finished with value: 0.38882650628355203 and parameters: {'taille_fenetre': 139, 'facteur_std': 2.5286967957383246}. Best is trial 4 with value: 0.38882650628355203.
[I 2

#### LT_moy_band avec HD

In [85]:
# ==========================================
# 1. LA FONCTION OBJECTIF POUR OPTUNA
# ==========================================
def objective(trial):
    """ 
    cette fonction suggère des val de param à tester -> donne le f1_score avec ce param 
    """
    # 1. Optuna suggère les paramètres à tester pour cet essai
    taille_fenetre_test = trial.suggest_int("taille_fenetre", 10, 150)
    facteur_std_test = trial.suggest_float("facteur_std", 1.0, 4.0)
    num_bandes_test = trial.suggest_int("num_bandes", 1, 10)
    
    # 2. On prépare notre fonction de détection avec ces nouveaux paramètres
    # On utilise partial pour "geler" les paramètres sans exécuter la fonction
    methode_test = partial(
        local_threshold_bandes, 
        taille_fenetre=taille_fenetre_test,
        facteur_std=facteur_std_test,
        num_bandes=num_bandes_test,
        metrique='Moyenne',
        visual_graph=False # Surtout pas de graphes pendant l'optimisation !
    )
    
    # 3. On appelle LA NOUVELLE fonction d'évaluation ULTIME (en mode silencieux)
    precision, recall, f1_score = evaluate_type_detection(
        methode_detection=methode_test,
        folder='train',
        img_type='HD',
        sequences=['sequence_1', 'sequence_2', 'sequence_3'], 
        dyns=['low dyn with columns 1', 'low dyn with columns 2', 'low dyn with columns 3'],
        update=False # <-- On cache les prints pour ne pas polluer le terminal
    )
    
    # 4. On renvoie uniquement le F1-score à Optuna car c'est ce qu'il doit maximiser
    return f1_score


# ==========================================
# 2. LANCEMENT DE L'OPTIMISATION
# ==========================================
if __name__ == "__main__":
    print("🚀 Lancement de l'optimisation Bayésienne TOTALE avec Optuna...")
    
    # On crée l'étude en demandant de maximiser la valeur renvoyée (le f1_score)
    study = optuna.create_study(direction="maximize")
    
    # On lance 50 essais avec la barre de progression activée
    study.optimize(objective, n_trials=50, show_progress_bar=True)

    # ==========================================
    # 3. AFFICHAGE DU RÉSULTAT FINAL
    # ==========================================
    print("\n" + 50*"🌟")
    print("OPTIMISATION TERMINÉE")
    print(50*"🌟")
    print(f"Meilleur F1-Score GLOBAL atteint : {study.best_value * 100:.2f}%")
    print("Paramètres parfaits pour ce score :")
    for cle, valeur in study.best_params.items():
        print(f"  -> {cle} : {valeur}")

[I 2026-06-01 20:32:13,045] A new study created in memory with name: no-name-9ad82c66-8214-4cb8-919d-f30f156fbe94


🚀 Lancement de l'optimisation Bayésienne TOTALE avec Optuna...


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-06-01 20:32:56,239] Trial 0 finished with value: 0.3526717943527983 and parameters: {'taille_fenetre': 117, 'facteur_std': 3.7305079343858005, 'num_bandes': 9}. Best is trial 0 with value: 0.3526717943527983.
[I 2026-06-01 20:33:35,958] Trial 1 finished with value: 0.4006174402365221 and parameters: {'taille_fenetre': 47, 'facteur_std': 1.8038005811057318, 'num_bandes': 4}. Best is trial 1 with value: 0.4006174402365221.
[I 2026-06-01 20:34:15,673] Trial 2 finished with value: 0.39055700834117146 and parameters: {'taille_fenetre': 34, 'facteur_std': 2.2130360222707717, 'num_bandes': 6}. Best is trial 1 with value: 0.4006174402365221.
[I 2026-06-01 20:34:57,048] Trial 3 finished with value: 0.26200691505331564 and parameters: {'taille_fenetre': 124, 'facteur_std': 1.5551762452418423, 'num_bandes': 5}. Best is trial 1 with value: 0.4006174402365221.
[I 2026-06-01 20:35:39,072] Trial 4 finished with value: 0.3942151197719254 and parameters: {'taille_fenetre': 50, 'facteur_std': 2.

# fonction pour PE 

In [ ]:
def local_threshold_PE(img_def, t) : 
    """ 
    Fonction qui va detecter def d'img en utlisant LT avec param optimaux selon le type passé en entrée 
    args : 
    img_def : img avec le defaut
    t : type de l'image """
    


In [ ]:
THRESHOLD_BAND_PARAMS = {
    'VGA':  {'taille_fenetre': 20, 'metrique': 'Moyenne', 'facteur_std': 1.8, 'num_bandes': 1},
    'SXGA':   {'taille_fenetre': 29, 'metrique': 'Moyenne', 'facteur_std': 2.8, 'num_bandes': 7},
    'HD': {'taille_fenetre': 26, 'metrique': 'Moyenne', 'facteur_std': 2.8, 'num_bandes': 8}
}

# Update pour avoir début et fin du défaut
Coumes va me donner la sortie de son random forest sous forme de numpy nd array 


### Nouvelle fonction d'évaluation pixel par pixel au lieu de col par col pour calculer F1 score

In [98]:
def evaluate_detection_pixels(vrai_dict, predit_dict, printing=False):
    """
    Évalue la détection au niveau du PIXEL pour le Region Growing.
    """
    vrais_pixels = set()
    predits_pixels = set()

    # 1. Extraction des pixels de la vérité terrain (Ground Truth)
    for x_str, infos in vrai_dict.items():
        x = int(x_str)
        # On lit la liste des (y_start, y_stop) créée par ta fonction get_defect_coordinates
        for (y_start, y_stop) in infos['ycords']:
            for y in range(int(y_start), int(y_stop) + 1): 
                vrais_pixels.add((x, y))

    # 2. Extraction des pixels de nos prédictions
    for x, segments in predit_dict.items():
        x = int(x)
        for (y_start, y_stop) in segments:
            for y in range(int(y_start), int(y_stop) + 1):
                predits_pixels.add((x, y))

    # 3. Calculs via l'intersection des Sets
    vp = len(vrais_pixels.intersection(predits_pixels))
    fp = len(predits_pixels - vrais_pixels) # Pénalise le RMSE_ok
    fn = len(vrais_pixels - predits_pixels) # Pénalise le RMSE_def

    precision = vp / (vp + fp) if (vp + fp) > 0 else 0
    rappel = vp / (vp + fn) if (vp + fn) > 0 else 0
    f1_score = 2 * (precision * rappel) / (precision + rappel) if (precision + rappel) > 0 else 0
    
    if printing:
        print("\n--- RÉSULTATS (Niveau PIXEL) ---")
        print(f"Pixels VP (Bien trouvés)  : {vp}")
        print(f"Pixels FN (Oubliés)       : {fn}  <-- Impacte RMSE_DEF")
        print(f"Pixels FP (Débordements)  : {fp}  <-- Impacte RMSE_OK")
        print(f"F1-Score (Pixels)         : {f1_score*100:.1f}%")

    return vp, fp, fn, f1_score

### Mesure def avec find_peaks + Region Growing 
prend image brute et le fameux ndarray du random forest en entrée, pour en sortir le dictionnaire final de correction.

In [112]:

from scipy.signal import find_peaks

def mesurer_defauts_finaux_(image, colonnes_rf_array, multiplicateur_rupture=3.5, taille_lissage=11):
    """
    Prend le numpy array [14, 34, 256] du Random Forest, trouve les graines 
    avec find_peaks, et mesure le défaut avec le Region Growing.
    """
    height = image.shape[0]
    segments_etendus = []
    
    # Filtre pour lisser le signal avant de chercher les graines
    filtre = np.ones(taille_lissage) / taille_lissage

    # On s'assure que l'array du RF est bien un vecteur 1D itérable
    colonnes_rf = np.atleast_1d(colonnes_rf_array)

    for x in colonnes_rf:
        x = int(x)
        colonne = image[:, x].astype(np.float32)
        
        # =========================================================
        # 1. LE RADAR (find_peaks avec lissage)
        # =========================================================
        colonne_lissee = convolve1d(colonne, filtre, mode='reflect')
        mediane_col = np.median(colonne_lissee)
        signal_anomalie = np.abs(colonne_lissee - mediane_col)
        
        # Le seuil de bruit pour éviter de détecter de fausses graines
        seuil_bruit = 2 * np.std(signal_anomalie) 
        graines, _ = find_peaks(signal_anomalie, height=seuil_bruit, distance=10)
        
        if len(graines) == 0:
            graines = [np.argmax(signal_anomalie)]

        # =========================================================
        # 2. LE MÈTRE RUBAN (Region Growing)
        # =========================================================
        sauts_verticaux = np.abs(np.diff(colonne))
        bruit_normal = np.median(sauts_verticaux)
        ecart_sauts = np.std(sauts_verticaux)
        seuil_rupture = bruit_normal + (multiplicateur_rupture * ecart_sauts)

        for y_seed in graines:
            y_start = int(y_seed)
            y_end = int(y_seed)

            while y_start > 0:
                if np.abs(colonne[y_start] - colonne[y_start - 1]) < seuil_rupture:
                    y_start -= 1
                else:
                    break

            while y_end < (height - 1):
                if np.abs(colonne[y_end] - colonne[y_end + 1]) < seuil_rupture:
                    y_end += 1
                else:
                    break
                    
            segments_etendus.append([x, y_start, y_end])

    # On utilise la fonction fusionner_segments de ton bloc [36]
    return fusionner_segments(segments_etendus)

### VErsion + rapide en prenannt direct tous les arrays plutot que pixel par pixel 

In [114]:

from scipy.signal import find_peaks

def mesurer_defauts_finaux(image, colonnes_rf_array, multiplicateur_rupture=3.5, taille_lissage=11):
    """
    Version VECTORISÉE ULTRA-RAPIDE. 
    Remplace les boucles 'while' par une recherche NumPy instantanée.
    """
    height = image.shape[0]
    segments_etendus = []
    
    filtre = np.ones(taille_lissage) / taille_lissage
    colonnes_rf = np.atleast_1d(colonnes_rf_array)

    for x in colonnes_rf:
        x = int(x)
        colonne = image[:, x].astype(np.float32)
        
        # =========================================================
        # 1. LE RADAR (inchangé)
        # =========================================================
        colonne_lissee = convolve1d(colonne, filtre, mode='reflect')
        mediane_col = np.median(colonne_lissee)
        signal_anomalie = np.abs(colonne_lissee - mediane_col)
        
        seuil_bruit = 2 * np.std(signal_anomalie) 
        graines, _ = find_peaks(signal_anomalie, height=seuil_bruit, distance=10)
        
        if len(graines) == 0:
            graines = [np.argmax(signal_anomalie)]

        # =========================================================
        # 2. LE MÈTRE RUBAN (Version Flash ⚡)
        # =========================================================
        # On calcule TOUTES les différences de la colonne d'un coup
        diffs = np.abs(np.diff(colonne))
        
        bruit_normal = np.median(diffs)
        ecart_sauts = np.std(diffs)
        seuil_rupture = bruit_normal + (multiplicateur_rupture * ecart_sauts)

        # On repère TOUS les "murs" (les indices où le saut dépasse le seuil)
        murs = np.where(diffs >= seuil_rupture)[0]

        for y_seed in graines:
            # --- Chercher le mur du HAUT ---
            murs_haut = murs[murs < y_seed]
            if len(murs_haut) > 0:
                y_start = int(murs_haut[-1] + 1) # Le mur le plus proche au-dessus
            else:
                y_start = 0

            # --- Chercher le mur du BAS ---
            murs_bas = murs[murs >= y_seed]
            if len(murs_bas) > 0:
                y_end = int(murs_bas[0])         # Le mur le plus proche en-dessous
            else:
                y_end = height - 1
                
            segments_etendus.append([x, y_start, y_end])

    return fusionner_segments(segments_etendus)

### Optimisation des param de ma fonction pour mesurer les défauts 


#### PRéchargement des do des diff types 
(pour que ce soit + rapide dans otpimisation et pas avoir à les recgarger à chaque fois)

In [115]:
def get_data(type_data):
    """ 
    Récupère toutes les images d'un type spécifique (VGA, SXGA, HD) 
    et les associe à leur vérité terrain (Ground Truth).
    """
    sequences = ['sequence_1', 'sequence_2', 'sequence_3']
    dyns = ['low dyn with columns 1', 'low dyn with columns 2', 'low dyn with columns 3']
    dico_data = []

    print(f"\n📥 Pré-chargement des données {type_data} en mémoire...")

    for seq in sequences:
        for dyn in dyns:
            chiffre = int(dyn.split()[-1])
            json_name = f'{type_data}_{seq}_config_{chiffre}'
            chemin_json = f'results/{json_name}.json'
            
            try:
                # 1. Chargement lourd depuis le disque
                data = load_images(folder='train', type=type_data, sequence=seq, dyn=dyn, force_gray=False)
                json_data = load_json(chemin_json)
                
                # 2. Association en mémoire vive (RAM)
                for i in range(len(data)):
                    vrai_dict = get_defect_coordinates(json_data, i)
                    dico_data.append({
                        'image': data[i],
                        'vrai_dict': vrai_dict
                    })
            except FileNotFoundError:
                print(f"  ⚠️ Fichier introuvable, combinaison ignorée : {seq} / {dyn}")
                
    print(f"✅ {len(dico_data)} images prêtes pour l'optimisation du {type_data}.")
    
    return dico_data



# get_data('VGA')
# get_data('HD')
# get_data('SXGA')

In [116]:

# =========================================================
# LA FONCTION OBJECTIF (Adaptée pour recevoir les données)
# =========================================================
def objective_region_growing(trial, donnees_prechargees):
    """
    Fonction évaluée par Optuna. Elle prend l'objet 'trial' (qui suggère les paramètres)
    ET les données pré-chargées en mémoire pour éviter de lire le disque dur.
    """
    # Optuna suggère les paramètres à tester pour ce "tour" de boucle
    multiplicateur_test = trial.suggest_float("multiplicateur_rupture", 1.0, 6.0)
    lissage_test = trial.suggest_int("taille_lissage", 3, 21, step=2) # step=2 garantit un nombre impair
    
    f1_pixels_scores = []

    # On boucle sur la liste des images en RAM
    for item in donnees_prechargees:
        image = item['image']
        vrai_dict = item['vrai_dict']
        
        # --- LA RUSE (En attendant le Random Forest du collègue) ---
        # On extrait les vraies colonnes défectueuses pour simuler un RF parfait à 100%
        colonnes_simulees_rf = np.array(list(vrai_dict.keys()), dtype=int)
        
        # S'il n'y a aucun défaut sur cette image, on passe à la suivante
        if len(colonnes_simulees_rf) == 0:
            continue 
            
        # On lance ton algorithme de mesure des "Y" (trouver les graines + grandir)
        predit_dict = mesurer_defauts_finaux(
            image, 
            colonnes_rf_array=colonnes_simulees_rf, 
            multiplicateur_rupture=multiplicateur_test, 
            taille_lissage=lissage_test
        )
        
        # On évalue le score AU NIVEAU DU PIXEL pour vérifier qu'on ne déborde pas
        vp, fp, fn, f1_pixel = evaluate_detection_pixels(vrai_dict, predit_dict)
        f1_pixels_scores.append(f1_pixel)
        
    # Optuna a besoin d'une seule valeur finale à maximiser (la moyenne des scores)
    return np.mean(f1_pixels_scores) if f1_pixels_scores else 0.0



#### Optimisation 3 type en même temps  

In [ ]:

# =========================================================
# LANCEMENT DE L'OPTIMISATION
# =========================================================
if __name__ == "__main__":
    optuna.logging.set_verbosity(optuna.logging.WARNING) # pour avoir barre de progression propre 

    # Liste des types que tu veux optimiser
    types_a_optimiser = ['VGA', 'HD', 'SXGA']
    
    # Dictionnaire pour sauvegarder les meilleurs paramètres de chaque type
    parametres_finaux = {}

    for type_capteur in types_a_optimiser:
        print("\n" + 50*"=")
        print(f"🚀 DÉMARRAGE DE L'OPTIMISATION POUR : {type_capteur}")
        print(50*"=")
        
        # 1. On charge toutes les données de ce type en mémoire (ça prend quelques secondes)
        donnees = get_data(type_capteur)
        
        # 2. On crée une étude Optuna qui cherche à maximiser le F1-Score
        study = optuna.create_study(direction="maximize")
        
        # 3. L'ASTUCE DU LAMBDA : 
        # Optuna s'attend à une fonction du type f(trial). On utilise lambda pour 
        # lui glisser nos 'donnees' en douce !
        study.optimize(lambda trial: objective_region_growing(trial, donnees), n_trials=50, show_progress_bar=True)
        
        # 4. On sauvegarde les paramètres en or trouvés pour ce type
        parametres_finaux[type_capteur] = study.best_params
        
        print(f"\n🏆 Score max atteint pour {type_capteur} : {study.best_value * 100:.2f}%")


    # =========================================================
    # 4. LE RÉCAPITULATIF FINAL (À copier dans ton code final)
    # =========================================================
    print("\n\n" + 60*"🌟")
    print("BILAN DES MEILLEURS PARAMÈTRES À CODER EN DUR")
    print(60*"🌟")
    for type_capteur, params in parametres_finaux.items():
        print(f"\nParamètres optimisés pour le capteur {type_capteur} :")
        for cle, valeur in params.items():
            # Arrondi pour rendre ça plus propre à copier-coller
            if isinstance(valeur, float):
                print(f"  -> '{cle}': {valeur:.2f}")
            else:
                print(f"  -> '{cle}': {valeur}")


🚀 DÉMARRAGE DE L'OPTIMISATION POUR : VGA

📥 Pré-chargement des données VGA en mémoire...
✅ 5967 images prêtes pour l'optimisation du VGA.


  0%|          | 0/50 [00:00<?, ?it/s]


🏆 Score max atteint pour VGA : 80.46%

🚀 DÉMARRAGE DE L'OPTIMISATION POUR : HD

📥 Pré-chargement des données HD en mémoire...
✅ 2700 images prêtes pour l'optimisation du HD.


  0%|          | 0/50 [00:00<?, ?it/s]

[W 2026-06-02 16:19:16,175] Trial 4 failed with parameters: {'multiplicateur_rupture': 4.965119869553354, 'taille_lissage': 7} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/opt/anaconda3/envs/phelma/lib/python3.10/site-packages/optuna/study/_optimize.py", line 205, in _run_trial
    value_or_values = func(trial)
  File "/var/folders/ty/mz254c797ml4f8jydylf3h6c0000gn/T/ipykernel_96067/2475371470.py", line 70, in <lambda>
    study.optimize(lambda trial: objective_region_growing(trial, donnees), n_trials=50, show_progress_bar=True)
  File "/var/folders/ty/mz254c797ml4f8jydylf3h6c0000gn/T/ipykernel_96067/2475371470.py", line 29, in objective_region_growing
    predit_dict = mesurer_defauts_finaux(
  File "/var/folders/ty/mz254c797ml4f8jydylf3h6c0000gn/T/ipykernel_96067/3296585224.py", line 54, in mesurer_defauts_finaux
    if np.abs(colonne[y_end] - colonne[y_end + 1]) < seuil_rupture:
KeyboardInterrupt
[W 2026-06-02 16:19:16,213] Trial 4

KeyboardInterrupt: 

In [111]:
param_VGA = parametres_finaux
print("Mes paramètres sauvés :", param_VGA)

Mes paramètres sauvés : {'VGA': {'multiplicateur_rupture': 5.929219044423664, 'taille_lissage': 21}}


### Je relance pour SXGA

In [117]:
# =========================================================
# LANCEMENT DE L'OPTIMISATION
# =========================================================
if __name__ == "__main__":

    # 💡 ASTUCE : Ne mets qu'un seul capteur à la fois ici pour ne pas bloquer ton PC toute l'aprèm !
    types_a_optimiser = ['HD'] 
    
    parametres_finaux = {}

    for type_capteur in types_a_optimiser:
        print("\n" + 50*"=")
        print(f"🚀 DÉMARRAGE DE L'OPTIMISATION POUR : {type_capteur}")
        print(50*"=")
        
        donnees = get_data(type_capteur)
        study = optuna.create_study(direction="maximize")
        
        study.optimize(lambda trial: objective_region_growing(trial, donnees), n_trials=50, show_progress_bar=True)
        
        parametres_finaux[type_capteur] = study.best_params
        
        # ✅ CORRECTION : ON AFFICHE LES PARAMÈTRES IMMÉDIATEMENT DANS LA BOUCLE !
        print("\n" + 40*"*")
        print(f"🏆 Score max atteint pour {type_capteur} : {study.best_value * 100:.2f}%")
        print(f"👉 LES PARAMÈTRES SAUVÉS POUR {type_capteur} SONT :")
        for cle, valeur in study.best_params.items():
            if isinstance(valeur, float):
                print(f"  -> '{cle}': {valeur:.2f}")
            else:
                print(f"  -> '{cle}': {valeur}")
        print(40*"*" + "\n")

    # (Le récapitulatif final reste ici au cas où tu en lances plusieurs, mais n'est plus indispensable)


🚀 DÉMARRAGE DE L'OPTIMISATION POUR : HD

📥 Pré-chargement des données HD en mémoire...
✅ 2700 images prêtes pour l'optimisation du HD.


  0%|          | 0/50 [00:00<?, ?it/s]


****************************************
🏆 Score max atteint pour HD : 76.08%
👉 LES PARAMÈTRES SAUVÉS POUR HD SONT :
  -> 'multiplicateur_rupture': 3.84
  -> 'taille_lissage': 3
****************************************



### Pour SXGA 

In [118]:
# =========================================================
# LANCEMENT DE L'OPTIMISATION
# =========================================================
if __name__ == "__main__":

    # 💡 ASTUCE : Ne mets qu'un seul capteur à la fois ici pour ne pas bloquer ton PC toute l'aprèm !
    types_a_optimiser = ['SXGA'] 
    
    parametres_finaux = {}

    for type_capteur in types_a_optimiser:
        print("\n" + 50*"=")
        print(f"🚀 DÉMARRAGE DE L'OPTIMISATION POUR : {type_capteur}")
        print(50*"=")
        
        donnees = get_data(type_capteur)
        study = optuna.create_study(direction="maximize")
        
        study.optimize(lambda trial: objective_region_growing(trial, donnees), n_trials=50, show_progress_bar=True)
        
        parametres_finaux[type_capteur] = study.best_params
        
        # ✅ CORRECTION : ON AFFICHE LES PARAMÈTRES IMMÉDIATEMENT DANS LA BOUCLE !
        print("\n" + 40*"*")
        print(f"🏆 Score max atteint pour {type_capteur} : {study.best_value * 100:.2f}%")
        print(f"👉 LES PARAMÈTRES SAUVÉS POUR {type_capteur} SONT :")
        for cle, valeur in study.best_params.items():
            if isinstance(valeur, float):
                print(f"  -> '{cle}': {valeur:.2f}")
            else:
                print(f"  -> '{cle}': {valeur}")
        print(40*"*" + "\n")

    # (Le récapitulatif final reste ici au cas où tu en lances plusieurs, mais n'est plus indispensable)


🚀 DÉMARRAGE DE L'OPTIMISATION POUR : SXGA

📥 Pré-chargement des données SXGA en mémoire...
✅ 4764 images prêtes pour l'optimisation du SXGA.


  0%|          | 0/50 [00:00<?, ?it/s]


****************************************
🏆 Score max atteint pour SXGA : 74.37%
👉 LES PARAMÈTRES SAUVÉS POUR SXGA SONT :
  -> 'multiplicateur_rupture': 6.00
  -> 'taille_lissage': 15
****************************************



In [ ]:
PARAMETRES_REGION_GROWING = { 
'VGA': {'multiplicateur_rupture': 5.9,  'taille_lissage': 21},  
'SXGA': {'multiplicateur_rupture': 6,  'taille_lissage': 15},  
'HD': {'multiplicateur_rupture': 3.84,  'taille_lissage': 3} 
}



In [121]:
# =========================================================
# TEST DE VÉRIFICATION VISUELLE SUR UNE IMAGE VGA
# =========================================================

# 1. On charge les données d'un dossier VGA au hasard (ici sequence 1, dyn 3)
# (Assure-toi que les chemins correspondent à ton architecture)
data_test = load_images(folder='train', type='VGA', sequence='sequence_1', dyn='low dyn with columns 3')
json_test = load_json('results/VGA_sequence_1_config_3.json')

# 2. On prend la toute première image de ce dossier (l'image d'index 0)
image_test = data_test[0]

# 3. On lit le JSON pour savoir quelles sont les vraies colonnes défectueuses
vrai_dict = get_defect_coordinates(json_test, 0)
colonnes_simulees = np.array(list(vrai_dict.keys()), dtype=int)

print(f"L'image fait {image_test.shape[0]} pixels de haut.")
print(f"Colonnes données à l'algorithme : {colonnes_simulees}")

# 4. On lance TON algorithme avec les paramètres extrêmes trouvés par Optuna pour le VGA
dictionnaire_defauts_predit = mesurer_defauts_finaux(
    image_test, 
    colonnes_rf_array=colonnes_simulees, 
    multiplicateur_rupture=3.5,   # Le paramètre extrême
    taille_lissage=21             # Le paramètre extrême
)

# 5. On affiche le résultat !
print("\n" + 40*"=")
print("RÉSULTAT DU REGION GROWING :")
print(40*"=")
print(dictionnaire_defauts_predit)

L'image fait 512 pixels de haut.
Colonnes données à l'algorithme : [ 45 120 168 294 361 489 525]

RÉSULTAT DU REGION GROWING :
{45: [(0, 43), (44, 196), (256, 341), (342, 511)], 120: [(73, 156), (157, 299), (300, 392), (393, 483)], 168: [(57, 411), (412, 511)], 294: [(0, 167), (168, 373), (405, 458), (460, 511)], 361: [(0, 91), (92, 292), (293, 404), (447, 511)], 489: [(292, 511)], 525: [(0, 332), (333, 511)]}


In [120]:
print("\n" + 40*"=")
print("VÉRITÉ TERRAIN (Ce que dit le JSON) :")
print(40*"=")

# On affiche joliment les vraies coordonnées pour comparer
for col, vraies_coordonnees in vrai_dict.items():
    print(f"Colonne {col} : Le(s) vrai(s) défaut(s) sont en -> {vraies_coordonnees}")


VÉRITÉ TERRAIN (Ce que dit le JSON) :
Colonne 45 : Le(s) vrai(s) défaut(s) sont en -> {'ycords': [(159, 254), (342, 417), (448, 502)], 'type': 'blinking'}
Colonne 120 : Le(s) vrai(s) défaut(s) sont en -> {'ycords': [(0, 512)], 'type': 'noisy_blinking'}
Colonne 168 : Le(s) vrai(s) défaut(s) sont en -> {'ycords': [(0, 512)], 'type': 'noisy_blinking'}
Colonne 294 : Le(s) vrai(s) défaut(s) sont en -> {'ycords': [(0, 512)], 'type': 'blinking'}
Colonne 361 : Le(s) vrai(s) défaut(s) sont en -> {'ycords': [(0, 512)], 'type': 'noisy_blinking'}
Colonne 489 : Le(s) vrai(s) défaut(s) sont en -> {'ycords': [(409, 504)], 'type': 'noisy_blinking'}
Colonne 525 : Le(s) vrai(s) défaut(s) sont en -> {'ycords': [(0, 512)], 'type': 'noisy'}


### Nouvelle version siplifiée 